# Portfolio optimisation of the S&P 500 universe (about 500 US stocks): Markowitz, QUBO, simulated annealing, MIQP and the Ising spin-glass model

This notebook repeats, on the **S&P 500 stocks**, the study done for 20 stocks in `portfolio_optimization_20stocks.ipynb`: same data, same $\mu$, $\Sigma$, $\lambda$, same QUBO, same objective for every solver. Run it **once from top to bottom**. The data are downloaded **only once**, in the first code cell.

## Objectives and where they are done

| # | Objective | Section |
|---|---|---|
| 1 | Daily adjusted closing prices of the S&P 500 stocks from the Yahoo Finance API | Section 1 |
| 2 | Sharpe-ratio chart and Markowitz model (values $\mu$, $\Sigma$, $\lambda$ used in the QUBO) | Sections 2, 3, 4, 5 |
| 3 | Classical solvers on the QUBO: simulated annealing and MIQP | Sections 6 and 7 |
| 4 | Ising spin-glass model and its optimisation | Section 8 |
| 5 | Comparison table (simulated annealing / MIQP versus Ising) and written statement | Section 9 |

## What is different from the 20-stock notebook, and why

With about 500 stocks the problem is $2^{500}$ times larger, so some steps of the 20-stock notebook cannot be done in the same way. Nothing below is an approximation of the model; only the way it is computed or checked changes.

| 20 stocks | About 500 stocks |
|---|---|
| Exhaustive search over all $\binom{20}{k}$ portfolios gives the **exact optimum** | $\binom{N}{k}$ is astronomically large, so there is **no exact optimum**. Results are compared with the **best known** objective (lowest value found by any solver) and with the **MIQP lower bound** |
| QUBO solved for every $k=1\ldots 20$ | QUBO solved for a **list of $k$ values** `K_LIST` (Section 4.1); $k^\star$ is the $k$ with the lowest objective, as before |
| Convex Markowitz problems solved with SLSQP | SLSQP is far too slow for 500 variables, so the **same convex quadratic programmes** are solved with an exact convex QP solver (`cvxpy` + Clarabel) |
| MIQP with Gurobi (`pip` licence) | The `pip` Gurobi licence is limited to a few hundred variables, so the MIQP is solved by SCIP (free) after an **exact eigenvalue reformulation** of $\Sigma$ (Section 7), or by Gurobi if you have a full licence. A time limit is set and the optimality gap is reported |
| Validation of Ising = QUBO on all $2^{20}$ states | Validation on **random** states (all $2^{N}$ are impossible) |
| 30 seeds, 1000 reads x 1000 sweeps | Fewer seeds and reads (settings in Section 4.1), because one 1000 x 1000 run on 500 spins takes about a minute |

## Sections

1. **Data**: S&P 500 ticker list, download of the adjusted closing prices, keep stocks with full history.
2. **Sharpe-ratio charts**.
3. **Markowitz model chart**.
4. **Settings, inputs and capped Markowitz solutions**.
5. **QUBO formulation**: $\lambda$, QUBO matrix, Monte-Carlo penalty $P$.
6. **Classical solver 1, simulated annealing**: own annealer for each $k$, choice of $k^\star$, D-Wave sampler.
7. **Classical solver 2, MIQP**.
8. **Ising spin-glass model**.
9. **Comparison table and conclusions**.

Packages: `numpy pandas scipy matplotlib yfinance cvxpy dimod dwave-samplers pyscipopt` (and optionally `gurobipy` with a full licence). Missing packages are installed automatically by `_ensure`. Figures and tables are saved in `portfolio_500stocks_outputs`.

## Section 1. Data: S&P 500 stocks, daily adjusted closing prices (Yahoo Finance API)

**What the next cell does (objective 1)**

1. imports all libraries and defines the helper `_ensure` (installs a missing package);
2. reads the list of **current S&P 500 constituents** from the public file `constituents.csv` of the `datasets/s-and-p-500-companies` repository (column `Symbol`; the file also gives the date each stock was added to the index; the newest date in the file is printed so you can see how recent the list is). Tickers are converted to the Yahoo format (`BRK.B` becomes `BRK-B`). You can use your own list instead by setting `CONSTITUENTS_CSV` to a CSV file with a `Symbol` column;
3. downloads the daily **adjusted close** (`Adj Close`: split- and dividend-adjusted) of every ticker with `yfinance` from 2015-01-01 to 2025-12-31 (the `end` date is exclusive), in chunks of 50 tickers with retries; tickers that return nothing are retried one by one;
4. keeps only the stocks that have a price on **every** trading day of the period (**full history**). Stocks that started trading after 2015-01-01, or that have missing days, are dropped; the notebook prints how many stocks are kept and dropped and lists the dropped ones with the reason (first price date, missing days);
5. saves the table once as `us_stock_data_500/adj_close_wide_500.csv`.

The table `adj_close` (rows = dates, columns = the kept tickers) is created **only here** and used by every later cell.

In [ ]:
# SECTION 1 | DATA - the only cell that downloads data. Creates the table `adj_close` used by the whole notebook.
import os, sys, time, subprocess, importlib, itertools, warnings
from math import comb
from itertools import combinations, islice

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy.optimize import minimize
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

def _ensure(module, pip_name):
    """import a package; if it is missing, install it into the Python that runs THIS notebook, then import it."""
    try:
        return importlib.import_module(module)
    except ImportError:
        print(f"'{module}' not found in {sys.executable} -> installing '{pip_name}' ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        importlib.invalidate_caches()
        return importlib.import_module(module)

yf = _ensure("yfinance", "yfinance")

START_DATE = "2015-01-01"
END_DATE = "2026-01-01"          # yfinance 'end' is exclusive
CONSTITUENTS_URL = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv"
CONSTITUENTS_CSV = None          # optional: path of your own csv with a column 'Symbol' (used instead of the URL)
CHUNK = 50                       # tickers per download request
MAX_RETRIES = 3
DATA_DIR = "us_stock_data_500"
OUT_DIR = "portfolio_500stocks_outputs"       # every figure / table of the notebook is saved here
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

def savefig(name, dpi=300):
    plt.savefig(f"{OUT_DIR}/{name}.png", dpi=dpi, bbox_inches="tight")

# ------------------------------- S&P 500 ticker list -------------------------------
cons = pd.read_csv(CONSTITUENTS_CSV if CONSTITUENTS_CSV else CONSTITUENTS_URL)
cons["Ticker"] = cons["Symbol"].astype(str).str.strip().str.replace(".", "-", regex=False)   # Yahoo uses BRK-B, not BRK.B
cons = cons.drop_duplicates("Ticker").reset_index(drop=True)
TICKERS = cons["Ticker"].tolist()
names = dict(zip(cons["Ticker"], cons["Security"] if "Security" in cons.columns else cons["Ticker"]))
print(f"Ticker list: {CONSTITUENTS_CSV if CONSTITUENTS_CSV else CONSTITUENTS_URL}")
print(f"{len(TICKERS)} tickers" + (f" | newest 'Date added' in the file: {pd.to_datetime(cons['Date added'], errors='coerce').max().date()}"
                                    if "Date added" in cons.columns else ""))

# --------------------------------- download ---------------------------------
def download_adj_close(tickers):
    """Daily adjusted close of a list of tickers (auto_adjust=False keeps the 'Adj Close' column), with retries."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            df = yf.download(list(tickers), start=START_DATE, end=END_DATE, interval="1d", auto_adjust=False,
                             progress=False, threads=True)
            if df is not None and not df.empty:
                adj = df["Adj Close"]
                if isinstance(adj, pd.Series):
                    adj = adj.to_frame(tickers[0])
                adj.index = pd.to_datetime(adj.index)
                return adj
            print(f"  empty result (attempt {attempt}/{MAX_RETRIES})")
        except Exception as e:
            print(f"  error: {e} (attempt {attempt}/{MAX_RETRIES})")
        time.sleep(2 * attempt)
    return pd.DataFrame()

parts = []
for i in range(0, len(TICKERS), CHUNK):
    chunk = TICKERS[i:i + CHUNK]
    print(f"downloading tickers {i + 1}-{i + len(chunk)} of {len(TICKERS)} ...")
    parts.append(download_adj_close(chunk))
    time.sleep(1.0)                                                   # be polite to Yahoo
adj_all = pd.concat([p for p in parts if not p.empty], axis=1)
adj_all = adj_all.loc[:, ~adj_all.columns.duplicated()]
empty_t = [t for t in TICKERS if t not in adj_all.columns or adj_all[t].isna().all()]
if empty_t:
    print(f"\n{len(empty_t)} tickers returned nothing in the chunks, retrying them one by one: {empty_t}")
    extra = [download_adj_close([t]) for t in empty_t]
    adj_all = pd.concat([adj_all.drop(columns=[t for t in empty_t if t in adj_all.columns])] + [e for e in extra if not e.empty], axis=1)
adj_all = adj_all.sort_index().dropna(how="all")                    # drop days on which no stock has a price

# ------------------------ keep only stocks with full history ------------------------
full_history = adj_all.notna().all()                                 # a price on every trading day of the period
kept = [t for t in TICKERS if t in adj_all.columns and bool(full_history[t])]            # keep the order of the ticker list
adj_close = adj_all[kept]
dropped = [t for t in TICKERS if t not in adj_close.columns]
drop_tab = pd.DataFrame({"Name": [names.get(t, t) for t in dropped],
                         "First price date": [adj_all[t].first_valid_index() if t in adj_all.columns else pd.NaT for t in dropped],
                         "Missing days": [int(adj_all[t].isna().sum()) if t in adj_all.columns else len(adj_all) for t in dropped],
                         "Reason": ["no data returned" if (t not in adj_all.columns or adj_all[t].isna().all())
                                    else ("started after the first trading day" if adj_all[t].first_valid_index() > adj_all.index[0]
                                          else "missing days") for t in dropped]},
                        index=pd.Index(dropped, name="Ticker"))
adj_close.to_csv(f"{DATA_DIR}/adj_close_wide_500.csv")                # saved once for your records; not read again
drop_tab.to_csv(f"{DATA_DIR}/dropped_tickers.csv")

print("\n" + "=" * 78)
print(f"S&P 500 tickers in the list      : {len(TICKERS)}")
print(f"kept (full history, {adj_close.index[0].date()} to {adj_close.index[-1].date()}): {adj_close.shape[1]}")
print(f"dropped                          : {len(dropped)}   ({drop_tab['Reason'].value_counts().to_dict()})")
print(f"adj_close: {adj_close.shape[0]} dates x {adj_close.shape[1]} tickers")
print("=" * 78)
if dropped:
    print("Dropped tickers (first 30):")
    display(drop_tab.head(30))

## Section 2. Sharpe-ratio charts

These charts describe each stock on its own, before any optimisation. They use the table `adj_close` of Section 1. The Sharpe ratio uses a risk-free rate `RF_DESC = 2%` (as in the descriptive charts of the 20-stock notebook); the optimisation sections use `RF = 0`.

**What the next cell does**

* computes from the daily simple returns $r_t$: annualised return $\bar r\cdot 252$, annualised volatility $s_r\sqrt{252}$ and the Sharpe ratio
$$\text{Sharpe}=\frac{\bar r\cdot 252-\text{RF\_DESC}}{s_r\sqrt{252}};$$
* **Figure 1**: return versus volatility of every stock, coloured by Sharpe ratio, with iso-Sharpe lines; the 10 stocks with the highest Sharpe ratio are labelled (labelling all stocks would be unreadable);
* **Figure 2**: histogram of the Sharpe ratios of all stocks, and the 20 highest and 20 lowest Sharpe ratios;
* prints a summary of the Sharpe ratios and the table of the 10 highest and 10 lowest.

In [ ]:
# SECTION 2 | Sharpe-ratio charts of the S&P 500 stocks (uses adj_close of Section 1, no new download)
RF_DESC = 0.02                      # annual risk-free rate used for the descriptive Sharpe ratios
TRADING_DAYS = 252

ret_all = adj_close.pct_change().iloc[1:]                 # daily simple returns (every kept stock has a price every day)
ann_ret = ret_all.mean() * TRADING_DAYS
ann_vol = ret_all.std() * np.sqrt(TRADING_DAYS)
sharpe = (ann_ret - RF_DESC) / ann_vol
metrics = pd.DataFrame({"Ann. Return (%)": ann_ret * 100, "Ann. Vol (%)": ann_vol * 100, "Sharpe Ratio": sharpe})

# Figure 1: return vs volatility coloured by Sharpe ratio, iso-Sharpe lines
fig, ax = plt.subplots(figsize=(13, 8.5))
sc = ax.scatter(metrics["Ann. Vol (%)"], metrics["Ann. Return (%)"], c=metrics["Sharpe Ratio"], cmap="RdYlGn", s=28,
                edgecolor="black", linewidths=0.3, alpha=0.9)
vmax = metrics["Ann. Vol (%)"].max() * 1.05
vv = np.linspace(0, vmax, 50)
for s_ in [0.5, 1.0, 1.5, 2.0]:
    ax.plot(vv, RF_DESC * 100 + s_ * vv, ls="--", lw=1, color="grey", alpha=0.7)
    ax.text(vmax, RF_DESC * 100 + s_ * vmax, f" Sharpe {s_}", fontsize=8, color="grey", va="center")
top10 = metrics.sort_values("Sharpe Ratio", ascending=False).head(10)
for t, r_ in top10.iterrows():
    ax.annotate(t, (r_["Ann. Vol (%)"], r_["Ann. Return (%)"]), xytext=(4, 4), textcoords="offset points", fontsize=8, fontweight="bold")
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("Annualised volatility (%)"); ax.set_ylabel("Annualised return (%)"); ax.grid(True, alpha=0.3)
plt.colorbar(sc, ax=ax, label=f"Sharpe ratio (RF = {RF_DESC:.0%})")
ax.set_title(f"Return, volatility and Sharpe ratio of {len(metrics)} S&P 500 stocks ({adj_close.index[0].date()} to {adj_close.index[-1].date()})",
             fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("sharpe_return_vs_volatility"); plt.show()

# Figure 2: histogram of the Sharpe ratios and the extremes
fig, axes = plt.subplots(1, 3, figsize=(20, 7), gridspec_kw={"width_ratios": [1.1, 1, 1]})
ax = axes[0]
ax.hist(metrics["Sharpe Ratio"], bins=40, color="#1f77b4", edgecolor="white")
ax.axvline(metrics["Sharpe Ratio"].mean(), color="red", ls="--", label=f"mean {metrics['Sharpe Ratio'].mean():.2f}")
ax.axvline(metrics["Sharpe Ratio"].median(), color="orange", ls="--", label=f"median {metrics['Sharpe Ratio'].median():.2f}")
ax.set_xlabel(f"Sharpe ratio (RF = {RF_DESC:.0%})"); ax.set_ylabel("number of stocks"); ax.legend(); ax.grid(True, alpha=0.3)
ax.set_title("Distribution of the Sharpe ratios", fontweight="bold")
for ax, part, col, ttl in [(axes[1], metrics.sort_values("Sharpe Ratio", ascending=False).head(20), "#2ca02c", "20 highest Sharpe ratios"),
                           (axes[2], metrics.sort_values("Sharpe Ratio").head(20), "#d62728", "20 lowest Sharpe ratios")]:
    ax.barh(part.index[::-1], part["Sharpe Ratio"][::-1], color=col, edgecolor="black", linewidth=0.4)
    ax.set_xlabel("Sharpe ratio"); ax.set_title(ttl, fontweight="bold"); ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout(); savefig("sharpe_distribution_and_ranking"); plt.show()

print(f"{len(metrics)} stocks | Sharpe ratio: mean {metrics['Sharpe Ratio'].mean():.3f}, median {metrics['Sharpe Ratio'].median():.3f}, "
      f"max {metrics['Sharpe Ratio'].max():.3f} ({metrics['Sharpe Ratio'].idxmax()}), min {metrics['Sharpe Ratio'].min():.3f} ({metrics['Sharpe Ratio'].idxmin()}) | "
      f"{(metrics['Sharpe Ratio'] > 0).sum()} stocks with Sharpe > 0")
print("10 highest and 10 lowest Sharpe ratios:")
display(pd.concat([metrics.sort_values("Sharpe Ratio", ascending=False).head(10), metrics.sort_values("Sharpe Ratio").head(10)[::-1]]).round(2))
metrics.to_csv(f"{OUT_DIR}/stock_metrics_sharpe.csv")

## Section 3. Markowitz mean-variance model

For $n$ stocks with annualised mean returns $\mu$ and covariance matrix $\Sigma$, a portfolio is a weight vector $w$ (long-only and fully invested: $w_i\ge 0$, $\sum_i w_i=1$) with

$$\text{expected return } R(w)=\mu^\top w,\qquad \text{variance } \sigma^2(w)=w^\top\Sigma w,\qquad \text{Sharpe}(w)=\frac{R(w)-\text{RF}}{\sigma(w)} .$$

* **Minimum-variance portfolio**: $\min_w\; w^\top\Sigma w$.
* **Maximum-Sharpe (tangency) portfolio**: $\max_w\; \dfrac{\mu^\top w-\text{RF}}{\sqrt{w^\top\Sigma w}}$.
* **Efficient frontier**: for target returns $R^\star$, $\min_w\; w^\top\Sigma w$ subject to $\mu^\top w=R^\star$.

### 3.0 The quadratic-programming solvers used in the whole notebook

With about 500 variables SLSQP (used for 20 stocks) is far too slow, so the same problems are solved **exactly as convex quadratic programmes** with `cvxpy` and the interior-point solver Clarabel. $\Sigma$ is positive definite, so with its Cholesky factor $\Sigma=CC^\top$ we have $w^\top\Sigma w=\lVert C^\top w\rVert^2$.

* `qp_min_variance(Sigma, cap, target)`: $\min \lVert C^\top w\rVert^2$ s.t. $\sum w=1,\;0\le w\le \text{cap}$ (and $\mu^\top w=\text{target}$ for the frontier).
* `qp_mean_variance(mu, Sigma, gamma, cap)`: $\min \tfrac{\gamma}{2}\lVert C^\top w\rVert^2-\mu^\top w$ (exactly the objective the QUBO encodes).
* `qp_max_sharpe(mu, Sigma, cap, rf)`: the ratio is not convex, but with $y=\kappa w$, $\kappa=\sum_i y_i$ the problem is **equivalent** to the convex QP $\min y^\top\Sigma y$ s.t. $\mu^\top y-\text{RF}\,\kappa=1,\;\sum y=\kappa,\;0\le y\le \text{cap}\,\kappa,\;\kappa\ge0$, and $w=y/\kappa$ (valid when some feasible portfolio has $\mu^\top w>\text{RF}$).
* `frontier_points`: the same long-only efficient frontier as in the 20-stock notebook.

The identity $\lambda=(R_t-\text{RF})/\sigma_t^2$ of Section 5.2 gives an independent numerical check of these solvers: the mean-variance QP must return the maximum-Sharpe portfolio.

In [ ]:
# SECTION 3.0 | Convex QP solvers (cvxpy + Clarabel) for the long-only Markowitz problems
cp = _ensure("cvxpy", "cvxpy")

def equal_weight(n):
    return np.full(n, 1.0 / n)

def _chol_factor(Sigma):
    """A with  w' Sigma w = ||A w||^2   (Sigma is positive definite: Sigma = C C', A = C')."""
    return np.linalg.cholesky(Sigma).T

def _solve(objective, constraints):
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.CLARABEL)
    if prob.status not in ("optimal", "optimal_inaccurate"):
        raise RuntimeError(f"QP solver status: {prob.status}")
    return prob

def _clean(w, cap):
    w = np.clip(np.asarray(w, dtype=float).ravel(), 0.0, cap)
    return w / w.sum()

def qp_min_variance(Sigma, cap=1.0, target=None, mu=None):
    """min w'Sigma w  s.t. sum(w) = 1, 0 <= w <= cap  [and mu'w = target]."""
    n = len(Sigma); A = _chol_factor(Sigma); w = cp.Variable(n)
    cons = [cp.sum(w) == 1, w >= 0, w <= cap]
    if target is not None:
        cons.append(mu @ w == target)
    _solve(cp.Minimize(cp.sum_squares(A @ w)), cons)
    return _clean(w.value, cap)

def qp_mean_variance(mu, Sigma, gamma, cap=1.0):
    """min gamma/2 w'Sigma w - mu'w  s.t. sum(w) = 1, 0 <= w <= cap."""
    n = len(mu); A = _chol_factor(Sigma); w = cp.Variable(n)
    _solve(cp.Minimize(0.5 * gamma * cp.sum_squares(A @ w) - mu @ w), [cp.sum(w) == 1, w >= 0, w <= cap])
    return _clean(w.value, cap)

def qp_max_sharpe(mu, Sigma, cap=1.0, rf=0.0):
    """Maximum-Sharpe (tangency) portfolio through the equivalent convex QP in y = kappa * w."""
    n = len(mu); A = _chol_factor(Sigma); y = cp.Variable(n); kappa = cp.Variable()
    _solve(cp.Minimize(cp.sum_squares(A @ y)),
           [mu @ y - rf * kappa == 1, cp.sum(y) == kappa, y >= 0, y <= cap * kappa, kappa >= 0])
    return _clean(y.value / kappa.value, cap)

def qp_frontier(mu, Sigma, cap=1.0, n_pts=60):
    """Long-only efficient frontier. Returns arrays (variance, return), variance ascending."""
    n = len(mu)
    w_min = qp_min_variance(Sigma, cap)
    order, w_hi, left = np.argsort(-mu), np.zeros(n), 1.0
    for i in order:                                    # highest return reachable under the cap
        take = min(cap, left); w_hi[i] = take; left -= take
        if left <= 1e-12: break
    pts = []
    for tr in np.linspace(mu @ w_min, mu @ w_hi, n_pts)[:-1]:
        try:
            w_ = qp_min_variance(Sigma, cap, target=tr, mu=mu)
            pts.append((w_ @ Sigma @ w_, tr))
        except RuntimeError:
            pass
    pts.append((w_hi @ Sigma @ w_hi, mu @ w_hi))
    pts = np.array(pts)
    pts = pts[np.argsort(pts[:, 0])]
    return pts[:, 0], np.maximum.accumulate(pts[:, 1])
print("QP solvers ready (cvxpy", cp.__version__ + ", solver Clarabel)")

### 3.1 Markowitz chart (long-only, weights between 0 and 1)

**What the next cell does (objective 2)**

* uses the dates on which every stock traded and computes $\mu$ (daily mean $\times 252$) and $\Sigma$ (daily covariance $\times 252$);
* solves the minimum-variance and maximum-Sharpe problems and the efficient frontier (60 points) with the QP solvers of Section 3.0 (`RF = 0`, weights between 0 and 1);
* draws 10 000 random long-only portfolios (Dirichlet weights) as the feasible set, the efficient frontier, all individual stocks (not labelled, there are too many; the stocks held by the maximum-Sharpe portfolio are labelled) and the minimum-variance and maximum-Sharpe portfolios: expected return versus variance;
* prints the statistics of the two optimal portfolios and the stocks they hold.

In [ ]:
# SECTION 3.1 | Markowitz mean-variance model (efficient frontier, minimum-variance and maximum-Sharpe portfolios)
ANALYSIS_START = "2015-01-01"
ANALYSIS_END   = "2026-01-01"
TD = 252
N_RANDOM_PORT = 10000
N_FRONTIER = 60
RF = 0.0                        # risk-free rate for max-Sharpe (0, as in the 20-stock notebook)

prices = adj_close.loc[ANALYSIS_START:ANALYSIS_END].sort_index()           # adj_close comes from Section 1 (no new download)
daily_returns = prices.pct_change().dropna(how="all").dropna()              # same dates for every stock
assets = list(daily_returns.columns)
n = len(assets)
returns_array = (daily_returns.mean() * TD).to_numpy()
covariance_array = (daily_returns.cov() * TD).to_numpy()
assert np.all(np.linalg.eigvalsh(covariance_array) > 0), "covariance matrix is not positive definite"
print(f"{n} stocks | {daily_returns.index[0].date()} to {daily_returns.index[-1].date()} | {len(daily_returns)} daily observations")

t0 = time.time()
w_minvar = qp_min_variance(covariance_array, 1.0)
w_maxsh = qp_max_sharpe(returns_array, covariance_array, 1.0, RF)
ret_minvar, var_minvar = float(w_minvar @ returns_array), float(w_minvar @ covariance_array @ w_minvar)
ret_maxsh, var_maxsh = float(w_maxsh @ returns_array), float(w_maxsh @ covariance_array @ w_maxsh)
sharpe_maxsh = (ret_maxsh - RF) / np.sqrt(var_maxsh)
f_var, f_ret = qp_frontier(returns_array, covariance_array, 1.0, N_FRONTIER)       # efficient frontier
print(f"QP solutions in {time.time() - t0:.1f} s")

# Random long-only portfolios (feasible set)
rng = np.random.default_rng(42)
rw = rng.dirichlet(np.ones(n), size=N_RANDOM_PORT)
r_ret = rw @ returns_array
r_var = ((rw @ covariance_array) * rw).sum(axis=1)
r_sharpe = (r_ret - RF) / np.sqrt(np.maximum(r_var, 1e-15))

# Plot: expected return vs variance
plt.figure(figsize=(13, 8))
sc = plt.scatter(r_var, r_ret, c=r_sharpe, cmap="viridis", s=8, alpha=0.25, label="Random long-only portfolios")
plt.scatter(np.diag(covariance_array), returns_array, color="black", s=14, zorder=4, label="Individual stocks")
for i in np.flatnonzero(w_maxsh > 0.01):
    plt.annotate(assets[i], (covariance_array[i, i], returns_array[i]), xytext=(5, 5), textcoords="offset points", fontsize=9)
plt.plot(f_var, f_ret, color="red", lw=2.5, label="Efficient frontier")
plt.scatter(var_minvar, ret_minvar, color="blue", marker="*", s=220, zorder=5, label="Minimum-variance portfolio")
plt.scatter(var_maxsh, ret_maxsh, color="darkorange", marker="*", s=220, zorder=5, label="Maximum-Sharpe portfolio")
plt.xlabel("Annualized Portfolio Variance"); plt.ylabel("Annualized Expected Return")
plt.title(f"Markowitz Model: Expected Return vs Variance ({n} S&P 500 stocks, {daily_returns.index[0].date()} to {daily_returns.index[-1].date()})")
plt.colorbar(sc, label="Sharpe Ratio"); plt.grid(True, alpha=0.3); plt.legend(loc="best"); plt.tight_layout()
savefig("markowitz_frontier"); plt.show()

port_stats_tab = pd.DataFrame({
    "Portfolio": ["Minimum variance", "Maximum Sharpe"],
    "Expected Return (%)": [ret_minvar * 100, ret_maxsh * 100],
    "Variance": [var_minvar, var_maxsh],
    "Volatility (%)": [np.sqrt(var_minvar) * 100, np.sqrt(var_maxsh) * 100],
    "Sharpe Ratio": [(ret_minvar - RF) / np.sqrt(var_minvar), sharpe_maxsh],
    "Stocks held (weight > 0.01%)": [int((w_minvar > 1e-4).sum()), int((w_maxsh > 1e-4).sum())],
})
weights = pd.DataFrame({"Minimum-variance weight (%)": w_minvar * 100, "Maximum-Sharpe weight (%)": w_maxsh * 100}, index=assets)
weights = weights[(weights > 0.01).any(axis=1)].sort_values("Maximum-Sharpe weight (%)", ascending=False)
print("Optimized portfolio statistics:")
display(port_stats_tab.round(4))
print(f"Optimized portfolio weights (only stocks with weight > 0.01%; {len(weights)} stocks, 15 largest by max-Sharpe weight shown):")
display(weights.head(15).round(2))
port_stats_tab.to_csv(f"{OUT_DIR}/markowitz_portfolio_statistics.csv"); weights.to_csv(f"{OUT_DIR}/markowitz_weights.csv")

## Section 4. Settings, inputs ($\mu$, $\Sigma$) and capped Markowitz solutions

### 4.1 Shared settings and inputs

**What the next cell does**

* defines every setting of the optimisation sections: analysis period, 252 trading days, `RF = 0`, weight cap `MAX_W = 30 %` (used for the efficient-frontier curve and for $\lambda$), own-annealer budget `SA_CHAINS` x `SA_SWEEPS`, seed, optional Ledoit-Wolf shrinkage (`SHRINK_COV`, off, as in the 20-stock notebook) and the x-axis of the charts;
* **settings that are specific to this notebook** (you can change them):
  * `K_LIST`: the numbers of stocks $k$ for which the QUBO is solved (Section 6.2). The 20-stock notebook solved every $k=1\ldots 20$; solving every $k=1\ldots N$ here would be too slow, so a list is used;
  * `MIQP_TIME_LIMIT`: time limit in seconds of the MIQP solver;
  * `NUM_READS`, `NUM_SWEEPS`: reads and sweeps of the D-Wave sampler for the main run (1000 x 1000 as in the 20-stock notebook);
  * `N_SEEDS`, `EXP_READS`, `EXP_SWEEPS`: number of seeds, and reads and sweeps per read of the repeated experiments of Sections 8.7 and 8.8 (the 20-stock notebook used 30 seeds and 1000 reads);
* takes the prices from `adj_close`, computes the daily returns, keeps only days on which every stock traded, and computes once the annualised $\mu$ and $\Sigma$ that **every** optimiser below receives; checks that $\Sigma$ is positive definite.

In [ ]:
# SECTION 4.1 | Settings, data and inputs (mu, Sigma) - computed ONCE from the full period
# Uses the table `adj_close` of Section 1.  Every optimiser below receives exactly the same mu and Sigma.

# ---------------------------------- settings ----------------------------------
START, END  = "2015-01-01", "2026-01-01"           # full period used for the single optimisation
TD          = 252                                  # trading days per year
RF          = 0.0                                  # risk-free rate used in the optimisation sections
MAX_W       = 0.30                                 # cap per stock (used for the efficient-frontier curve and for lambda)
SA_CHAINS   = 128                                  # parallel annealing chains of the own simulated annealer
SA_SWEEPS   = 150                                  # sweeps per chain
SEED        = 42
SHRINK_COV  = False                                # True -> Ledoit-Wolf shrinkage of the covariance matrix
X_AXIS      = "variance"                           # "variance" (like the Markowitz chart) or "volatility"

# settings specific to the large-universe notebook
K_LIST          = [5, 10, 15, 20, 25, 30, 40, 50, 60, 75, 100]   # numbers of stocks k for which the QUBO is solved
MIQP_TIME_LIMIT = 600                              # seconds
NUM_READS, NUM_SWEEPS = 1000, 1000                 # D-Wave sampler, main run
N_SEEDS = 10                                       # repetitions of the experiments of Sections 8.7 and 8.8
EXP_READS, EXP_SWEEPS = 100, 1000                  # reads and sweeps per read in those repeated experiments (D-Wave sampler)

# ------------------------------------ data ------------------------------------
prices = adj_close.sort_index().loc[START:END]
returns = prices.pct_change().iloc[1:]
n_rows = len(returns)
returns = returns.dropna()                          # only days on which every stock traded
tickers = list(returns.columns)
N = len(tickers)
K_LIST = [k for k in K_LIST if k <= N]

def ledoit_wolf_cov(X):
    """Ledoit-Wolf shrinkage towards a scaled identity (same result as scikit-learn's LedoitWolf)."""
    X = X - X.mean(axis=0)
    T, n = X.shape
    S = X.T @ X / T
    m = np.trace(S) / n
    d2 = np.sum((S - m * np.eye(n)) ** 2)
    b2 = min((np.sum(np.sum(X ** 2, axis=1) ** 2) - T * np.sum(S ** 2)) / T ** 2, d2)
    sh = b2 / d2 if d2 > 0 else 0.0
    return (1 - sh) * S + sh * m * np.eye(n)

mu = returns.mean().to_numpy() * TD                                   # annualised mean return (arithmetic)
Sigma = (ledoit_wolf_cov(returns.to_numpy()) if SHRINK_COV else returns.cov().to_numpy()) * TD
assert np.all(np.linalg.eigvalsh(Sigma) > 0), "covariance matrix is not positive definite"

print(f"{N} stocks | {returns.index[0].date()} to {returns.index[-1].date()} | {len(returns)} daily returns "
      f"({n_rows - len(returns)} rows dropped for missing values)")
print(f"risk-free {RF:.1%} | weight cap {MAX_W:.0%} | covariance: {'Ledoit-Wolf' if SHRINK_COV else 'sample'}")
print(f"k values for the QUBO: {K_LIST}")

def xs(var):
    """x-axis coordinate of a portfolio from its variance."""
    var = np.asarray(var, dtype=float)
    return var if X_AXIS == "variance" else np.sqrt(var)
XLABEL = "Annualised portfolio variance" if X_AXIS == "variance" else "Annualised volatility"

### 4.2 Capped Markowitz solutions (reference portfolios)

The same mean-variance model as in Section 3, now with the settings of Section 4.1 and a **cap of 30 % per stock**; solved with the QP solvers of Section 3.0. These are the classical reference the QUBO is compared with.

**What the next cell does**

* defines the objective `mv_objective` $=\tfrac{\gamma}{2}w^\top\Sigma w-\mu^\top w$ and the statistics `port_stats` (return, variance, volatility, Sharpe);
* computes equal weight over all stocks (EW), the minimum-variance portfolio and the maximum-Sharpe portfolio (long-only, fully invested, each weight at most `MAX_W`);
* computes the efficient frontier (`f_var`, `f_ret`) used in the later charts.

In [ ]:
# SECTION 4.2 | Classical Markowitz solutions (the reference the QUBO is compared with)
#   EW = equal weight | MinVar = minimum variance | MaxSharpe = tangency portfolio
# All are long-only, fully invested, capped at MAX_W (convex QPs of Section 3.0).

def mv_objective(w, mu, Sigma, gamma):
    return 0.5 * gamma * w @ Sigma @ w - mu @ w

def port_stats(w, mu=mu, Sigma=Sigma):
    """(expected return, variance, volatility, Sharpe)"""
    ret, var = float(mu @ w), float(w @ Sigma @ w)
    return ret, var, np.sqrt(var), (ret - RF) / np.sqrt(var)

t0 = time.time()
w_ew = equal_weight(N)
w_minvar = qp_min_variance(Sigma, MAX_W)
w_maxsh = qp_max_sharpe(mu, Sigma, MAX_W, RF)
t_cls = time.time() - t0
f_var, f_ret = qp_frontier(mu, Sigma, MAX_W)
print(f"max-Sharpe portfolio (cap {MAX_W:.0%}): {(w_maxsh > 1e-6).sum()} stocks with weight > 0 | largest weight {w_maxsh.max() * 100:.1f}%"
      f"{' (the cap is binding)' if w_maxsh.max() > MAX_W - 1e-6 else ' (the cap is not binding)'} | QP solutions in {t_cls:.1f} s")

## Section 5. QUBO formulation

### 5.1 The model

Select exactly $k$ of the $N$ stocks and give each selected stock the weight $1/k$. With binary variables $x_i\in\{0,1\}$ ($x_i=1$ if stock $i$ is held), $w=x/k$ and

$$f_k(x)=\frac{\lambda}{2}\,\frac{x^\top\Sigma x}{k^2}-\frac{\mu^\top x}{k},\qquad H(x)=f_k(x)+P\Big(\sum_i x_i-k\Big)^2 .$$

Because $x_i^2=x_i$, $(\sum_i x_i-k)^2=x^\top(\mathbf 1\mathbf 1^\top-2k\,I)x+k^2$, so $H(x)=x^\top Qx+\text{const}$ with

$$Q=\frac{\lambda}{2k^2}\,\Sigma-\operatorname{diag}\!\Big(\frac{\mu}{k}\Big)+P\,(\mathbf 1\mathbf 1^\top-2k\,I),\qquad \text{const}=P\,k^2 .$$

For every $x$ with exactly $k$ ones the penalty is zero and $H(x)=f_k(x)$.

### 5.2 Risk aversion $\lambda$

$\lambda$ is the value for which the continuous problem $\min_w \tfrac{\lambda}{2}w^\top\Sigma w-\mu^\top w$ (long-only, fully invested) has the **maximum-Sharpe portfolio** $w_t$ as its optimum: $\lambda=\dfrac{R_t-\text{RF}}{\sigma_t^{2}}$ (from the first-order conditions of the tangency portfolio, $\Sigma w_t=\frac{\sigma_t^2}{R_t-\text{RF}}(\mu-\text{RF}\,\mathbf 1)$). It depends only on $\mu$ and $\Sigma$.

**What the next cell does**: computes $\lambda$ from the capped maximum-Sharpe portfolio of Section 4.2 and checks numerically that the QP of the mean-variance model at this $\lambda$ returns the same portfolio (an independent check of the QP solvers; the 30 % cap must not be binding for the identity). It also defines `obj_w`, the objective of any portfolio, used by every solver below.

In [ ]:
# SECTION 5.2 | Risk aversion lambda = (R_t - RF) / sigma_t^2 from the maximum-Sharpe portfolio of Section 4.2
r_t, v_t, _, sh_t = port_stats(w_maxsh)
LAMBDA = (r_t - RF) / v_t
w_chk = qp_mean_variance(mu, Sigma, LAMBDA, MAX_W)
print(f"lambda = (R_t - RF) / sigma_t^2 = ({r_t:.4f} - {RF:.2f}) / {v_t:.5f} = {LAMBDA:.4f}")
print(f"check : mean-variance QP at this lambda vs the max-Sharpe portfolio: max |w difference| = {np.abs(w_chk - w_maxsh).max():.2e}")

# objective of an (equal-weight or free-weight) portfolio, used by every solver below
obj_w = lambda w: 0.5 * LAMBDA * float(w @ Sigma @ w) - float(mu @ w)

### 5.3 Penalty coefficient $P$ by Monte Carlo, for every $k$ in `K_LIST`

The penalty must be large enough that breaking the constraint never pays, but not larger than necessary. It is estimated with Algorithm 1 of the paper (Eq. 8), with $A=(1,\ldots,1)$ and $b=k$ (the same procedure as in the 20-stock notebook):

1. draw `MC_N` random binary vectors $x$ (the number of ones is uniform on $0\ldots N$, positions random);
2. $x_{\text{from}}$ = the sampled vector with the smallest constraint violation $|\mathbf 1^\top x-k|$ (ties: the one with the lowest $f_k$);
3. $x_{\text{to}}$ = every sampled vector with $f_k(x_{\text{to}})<f_k(x_{\text{from}})$ and a larger violation;
4. candidate $P=\dfrac{f_k(x_{\text{from}})-f_k(x_{\text{to}})}{(\mathbf 1^\top x_{\text{to}}-k)^2-(\mathbf 1^\top x_{\text{from}}-k)^2}$, the estimate is the **largest** candidate (a lower bound for $P$);
5. repeated `MC_RUNS` times; the **largest estimate over all runs** is the penalty $P_k$.

If no sampled vector gains by breaking the constraint, the estimate of that run is $P=0$.

**What the next cell does**: runs the estimate for every $k$ in `K_LIST`, prints the table (mean, standard deviation, minimum, median, maximum = $P$ used, number of runs with $P=0$) and draws a box plot of the estimates against $k$. The result is stored in the dictionary `P_of_k` and used in all later sections. (The only change to the 20-stock code is that $x^\top\Sigma x$ is computed with a matrix product instead of a three-operand `einsum`; the numbers are identical, it is much faster for 500 stocks.)

In [ ]:
# SECTION 5.3 | Monte Carlo estimate of the QUBO penalty coefficient P for every k in K_LIST
# (Algorithm 1 of the paper, Eq. 8, with A = (1,...,1), b = k; see the markdown above)

MC_N    = 5000                         # random binary vectors per Monte Carlo run
MC_RUNS = 200                          # independent repetitions (gives the spread shown in the chart)
MC_SEED = SEED + 7

# Monte Carlo
def mc_binary_samples(n, m, rng):
    """m random binary vectors; the number of ones of each vector is uniform on 0..n, the positions are random."""
    card = rng.integers(0, n + 1, size=m)
    rank = np.argsort(np.argsort(rng.random((m, n)), axis=1), axis=1)
    return (rank < card[:, None]).astype(float)

def penalty_lower_bound(q, lin, s, k, lam):
    """Algorithm 1 for one sample set. q = x'Sigma x, lin = mu'x, s = sum(x) for every sampled vector."""
    f = 0.5 * lam / k ** 2 * q - lin / k             # x'Hx  (objective part of the QUBO)
    viol = (s - k) ** 2                              # (A x - b)^2
    v_min = viol.min()
    f_from = f[viol == v_min].min()                  # x_from
    m = (f < f_from) & (viol > v_min)                # x_to  (positive denominator)
    if not m.any():
        return 0.0          # no sampled vector gains by breaking the constraint -> this run needs no penalty (P = 0)
    return ((f_from - f[m]) / (viol[m] - v_min)).max()          # Eq. 8, largest candidate

rng_mc = np.random.default_rng(MC_SEED)
MC_P = np.zeros((MC_RUNS, len(K_LIST)))
t0 = time.time()
for run in range(MC_RUNS):
    Xs = mc_binary_samples(N, MC_N, rng_mc)
    q, lin, s = ((Xs @ Sigma) * Xs).sum(axis=1), Xs @ mu, Xs.sum(axis=1)       # x'Sigma x, mu'x, sum(x) of every sampled vector
    for j, k in enumerate(K_LIST):
        MC_P[run, j] = penalty_lower_bound(q, lin, s, k, LAMBDA)
print(f"Monte Carlo finished in {time.time() - t0:.1f} s  ({MC_RUNS} runs x {MC_N} random binary vectors, k in {K_LIST})")

P_K = MC_P.max(axis=0)                 # largest lower bound found for every k  -> penalty used by the next cells
P_of_k = dict(zip(K_LIST, P_K))
no_gain = [k for k, p in zip(K_LIST, P_K) if p == 0]
if no_gain:
    print(f"k = {no_gain}: in none of the {MC_RUNS} runs did any sampled vector gain by breaking the constraint, so the estimated penalty is P = 0 for these k.\n")

P_TABLE = pd.DataFrame({
    "k": K_LIST,
    "P mean": MC_P.mean(axis=0), "P std": MC_P.std(axis=0, ddof=1),
    "P min": MC_P.min(axis=0), "P median": np.median(MC_P, axis=0),
    "P max (used)": P_K,
    "runs with P = 0 (no violation gains)": (MC_P == 0).sum(axis=0),
}).set_index("k")
display(P_TABLE.round(6))
P_TABLE.to_csv(f"{OUT_DIR}/mc_penalty_vs_k.csv")

fig, ax = plt.subplots(figsize=(14, 7))
pos_ = np.arange(len(K_LIST))
bp = ax.boxplot([MC_P[:, j] for j in range(len(K_LIST))], positions=pos_, widths=0.6, patch_artist=True, showfliers=True,
                boxprops=dict(facecolor="#9ecae1", edgecolor="#08519c"), medianprops=dict(color="#d62728", lw=2),
                flierprops=dict(marker="o", ms=3, markerfacecolor="grey", markeredgecolor="grey", alpha=0.6))
ax.plot(pos_, MC_P.mean(axis=0), color="black", marker="o", ms=4, lw=1.2, label="Mean of the estimates")
ax.plot(pos_, P_K, color="#2ca02c", marker="^", ms=6, lw=1.2, ls="--", label="Largest estimate over all runs (penalty used for the QUBO)")
ax.set_xticks(pos_); ax.set_xticklabels(K_LIST)
ax.set_xlabel("k  (number of stocks selected)"); ax.set_ylabel("Estimated penalty coefficient P (lower bound, Eq. 8)")
ax.grid(True, alpha=0.3); ax.legend(loc="best")
ax.set_title(f"Monte Carlo estimate of the QUBO penalty P versus k  ({MC_RUNS} runs x {MC_N} random binary vectors, lambda = {LAMBDA:.3f})\n"
             f"box = quartiles, red line = median, whiskers/points = spread over repeated runs", fontsize=13, fontweight="bold")
plt.tight_layout(); savefig("mc_penalty_vs_k"); plt.show()

## Section 6. Classical solver 1: simulated annealing on the QUBO

### 6.1 The own simulated annealer and its helper functions

Simulated annealing minimises $x^\top Qx$ over $x\in\{0,1\}^N$. A move is accepted with the Metropolis rule: always if $\Delta E\le 0$, otherwise with probability $e^{-\Delta E/T}$. The temperature falls geometrically from $T_{\text{hi}}$ to $10^{-4}T_{\text{hi}}$, with $T_{\text{hi}}=0.1\times$ the median $|\Delta E|$ of single flips of random states. Each sweep proposes $N$ **single-bit flips** and $N$ **two-bit flips** (a swap of one selected for one unselected stock, which keeps the number of stocks).

**What the next cell defines**

* `simulated_annealing`: the same annealer as in the 20-stock notebook (many independent chains in parallel);
* `repair_select`: if a chain ends with the wrong number of stocks, stocks are added or removed one at a time, always taking the choice with the best objective $f(w)=\tfrac\lambda2 w^\top\Sigma w-\mu^\top w$ with $w=x/k$. This is the **same greedy rule as `repair_units` of the 20-stock notebook**, but the change of the objective for *all* candidates is computed at once from the exact formula $f(w\pm e_i/k)-f(w)=\pm g_i/k+\tfrac{\lambda}{2}\Sigma_{ii}/k^2$ with $g=\lambda\Sigma w-\mu$, which is much faster for 500 stocks and gives the same choice;
* `decode_select`: among the 10 lowest-energy chains it returns the best equal-weight portfolio (after repair if needed) and the share of chains that already hold exactly $k$ stocks;
* `build_qubo(k, P)`: the QUBO matrix $Q$ and constant of Section 5.1.

There is no `exact_select`: exhaustive search over all $\binom{N}{k}$ portfolios is impossible here.

In [ ]:
# SECTION 6.1 | QUBO solver - simulated annealing (pure NumPy, no extra packages)
#   simulated_annealing : many parallel chains, single-bit and two-bit flips
#   repair_select       : if a chain ends with the wrong number of stocks, add / remove single stocks greedily (best objective)
#   decode_select       : bits -> equal-weight portfolio
#   build_qubo          : QUBO matrix and constant of Section 5.1

def simulated_annealing(Q, n_chains=128, n_sweeps=150, rng=None, t_hi=None, t_lo=None, t_scale=0.1):
    """Minimise x'Qx over x in {0,1}^n. Returns final states (chains x n), energies (chains), (temperatures, mean energy per sweep).
    Each sweep proposes n single-bit flips and n two-bit flips (the two-bit moves swap one stock / weight unit for another)."""
    rng = rng if rng is not None else np.random.default_rng(SEED)
    Q = 0.5 * (Q + Q.T)
    n = Q.shape[0]
    d = np.diag(Q).copy()
    off = Q - np.diag(d)
    X = rng.integers(0, 2, size=(n_chains, n)).astype(float)
    H = X @ off
    E = np.einsum("ri,ri->r", X @ Q, X)
    if t_hi is None:                                   # start temperature: a fraction of the typical single-flip energy change
        t_hi = t_scale * np.median(np.abs((1 - 2 * X) * (d + 2 * H)))
    if t_lo is None:
        t_lo = t_hi * 1e-4
    temps = np.geomspace(t_hi, t_lo, n_sweeps)
    trace = np.empty(n_sweeps)
    for s, T in enumerate(temps):
        U1 = rng.random((n, n_chains))
        for j, i in enumerate(rng.permutation(n)):                      # single-bit flips
            delta = 1 - 2 * X[:, i]
            dE = delta * (d[i] + 2 * H[:, i])
            acc = (dE <= 0) | (U1[j] < np.exp(-np.maximum(dE, 0) / T))
            if acc.any():
                m = acc * delta
                X[:, i] += m; H += m[:, None] * off[i]; E += acc * dE
        U2 = rng.random((n, n_chains))
        pairs = np.array([rng.choice(n, 2, replace=False) for _ in range(n)])
        for j, (i, k) in enumerate(pairs):                              # two-bit flips
            di, dk = 1 - 2 * X[:, i], 1 - 2 * X[:, k]
            dE = di * (d[i] + 2 * H[:, i]) + dk * (d[k] + 2 * H[:, k]) + 2 * off[i, k] * di * dk
            acc = (dE <= 0) | (U2[j] < np.exp(-np.maximum(dE, 0) / T))
            if acc.any():
                mi, mk = acc * di, acc * dk
                X[:, i] += mi; X[:, k] += mk
                H += mi[:, None] * off[i] + mk[:, None] * off[k]; E += acc * dE
        trace[s] = E.mean()
    return X, np.einsum("ri,ri->r", X @ Q, X), (temps, trace)

def repair_select(x, k, mu, Sigma, gamma):
    """Greedy repair of a bit string to exactly k ones (weights x/k): add / remove ONE stock at a time, always taking the best
    objective f(w) = gamma/2 w'Sigma w - mu'w.  f(w -+ e_i/k) - f(w) is computed exactly for all candidates at once."""
    x = np.asarray(x, dtype=float).copy()
    d2 = np.diag(Sigma)
    while x.sum() != k:
        g = gamma * (Sigma @ (x / k)) - mu                        # gradient of f at w = x/k
        if x.sum() > k:                                            # remove the stock whose removal gives the lowest objective
            cand = np.flatnonzero(x > 0)
            x[cand[int(np.argmin(-g[cand] / k + 0.5 * gamma * d2[cand] / k ** 2))]] = 0.0
        else:                                                      # add the stock whose addition gives the lowest objective
            cand = np.flatnonzero(x == 0)
            x[cand[int(np.argmin(g[cand] / k + 0.5 * gamma * d2[cand] / k ** 2))]] = 1.0
    return x

def decode_select(X, E, q, mu, Sigma, gamma, top=10):
    k = q["k"]
    f = lambda xx: mv_objective(xx / k, mu, Sigma, gamma)
    best_x, best_f = None, np.inf
    for r in np.argsort(E)[:top]:
        x = repair_select(X[r], k, mu, Sigma, gamma)
        if f(x) < best_f:
            best_x, best_f = x, f(x)
    return best_x / k, dict(feasible_rate=float((X.sum(axis=1) == k).mean()), units=best_x)

def build_qubo(k, Pk):
    """QUBO matrix and constant of Section 5.1 for k stocks and penalty Pk:  H(x) = x'Qx + const."""
    Qk = 0.5 * LAMBDA / k ** 2 * Sigma - np.diag(mu / k) + Pk * (np.ones((N, N)) - 2 * k * np.eye(N))
    return Qk, Pk * k ** 2

### 6.2 QUBO solved for every $k$ in `K_LIST` (own simulated annealer)

**What the next cell does**

For each $k$ in `K_LIST` it builds $Q$ with the penalty $P_k$ of Section 5.3, anneals `SA_CHAINS` chains for `SA_SWEEPS` sweeps, decodes the best portfolio and checks that the QUBO energy of the chosen bit string equals the Markowitz objective of the equal-weight portfolio (no penalty is active). It prints one line per $k$ and a table: return, variance, volatility, Sharpe ratio, objective, share of chains with exactly $k$ stocks, lowest chain energy, penalty $P$ used and runtime. No exact optimum is available for comparison (Section 6.3 and the next sections use the best known value and the MIQP bound instead).

In [ ]:
# SECTION 6.2 | QUBO solution for every k in K_LIST (own simulated annealer)
rows, SOL = [], {}
for k in K_LIST:
    Pk = P_of_k[k]
    Qk, Ck = build_qubo(k, Pk)
    t0 = time.time()
    Xk, Ek, _ = simulated_annealing(Qk, SA_CHAINS, SA_SWEEPS, rng=np.random.default_rng(SEED + k))
    dt = time.time() - t0
    wk, info = decode_select(Xk, Ek, dict(k=k), mu, Sigma, LAMBDA)
    xk = np.round(wk * k)
    assert xk.sum() == k and np.isclose(xk @ Qk @ xk + Ck, obj_w(wk), atol=1e-9), "QUBO energy != Markowitz objective"
    SOL[k] = wk
    r_, v_, s_, h_ = port_stats(wk)
    rows.append({"k": k, "Return %": r_ * 100, "Variance": v_, "Vol %": s_ * 100, "Sharpe": h_,
                 "Objective (QUBO/SA)": obj_w(wk), "Chains with exactly k stocks (%)": info["feasible_rate"] * 100,
                 "Lowest chain energy": float(Ek.min() + Ck), "P used": Pk, "Runtime (s)": dt})
    print(f"k = {k:>3}: objective {obj_w(wk):.6f} | Sharpe {h_:.3f} | chains with exactly k stocks {info['feasible_rate'] * 100:5.1f}% | "
          f"P = {Pk:.5f} | {dt:.1f} s")
QK = pd.DataFrame(rows).set_index("k")
print()
display(QK.round(5))
QK.to_csv(f"{OUT_DIR}/qubo_sa_all_k.csv")

### 6.3 Choice of $k$ and the selected portfolio

**How $k$ is chosen**: the chosen $k^\star$ is the $k$ in `K_LIST` whose QUBO solution has the **lowest value of the same risk-return objective** $\tfrac{\lambda}{2}w^\top\Sigma w-\mu^\top w$ (the rule of the 20-stock notebook). Because the annealer is a heuristic and `K_LIST` is a list, $k^\star$ is the best value **found**; if $k^\star$ is the first or last value of `K_LIST` the notebook prints a warning (the list should then be extended). For orientation the number of stocks held by the continuous maximum-Sharpe portfolio is printed. From here on `K = K_STAR` in all solvers.

**What the next cell does**

* finds $k^\star$ and prints the selected stocks (equal weight $1/k^\star$) with their stand-alone return, volatility and Sharpe ratio;
* compares the QUBO selection with equal weight over all stocks and the Markowitz maximum-Sharpe and minimum-variance portfolios (same $\mu$, $\Sigma$, $\lambda$ for every row);
* **Chart**: risk-return map of all stocks with the efficient frontier, the QUBO solution of every $k$ and the choice $k^\star$ (left), and the objective and the Sharpe ratio as functions of $k$ (right).

In [ ]:
# SECTION 6.3 | choice of k and the selected portfolio
K_STAR = int(QK["Objective (QUBO/SA)"].idxmin())
w_star = SOL[K_STAR]
sel = np.flatnonzero(w_star)
print("=" * 78)
print(f"QUBO RESULT (own SA): k* = {K_STAR} of {N} stocks, equal weight {100 / K_STAR:.2f}% each")
print("=" * 78)
if K_STAR in (min(K_LIST), max(K_LIST)):
    print(f"WARNING: k* = {K_STAR} is the {'first' if K_STAR == min(K_LIST) else 'last'} value of K_LIST; extend K_LIST in Section 4.1 to check whether a larger / smaller k is better.")
print(f"for orientation: the continuous maximum-Sharpe portfolio (cap {MAX_W:.0%}) holds {(w_maxsh > 1e-6).sum()} stocks with weight > 0")

sig_ = np.sqrt(np.diag(Sigma))
stock_tab = pd.DataFrame({"Annualised return (%)": mu[sel] * 100, "Volatility (%)": sig_[sel] * 100,
                          "Stand-alone Sharpe": (mu[sel] - RF) / sig_[sel]}, index=[tickers[i] for i in sel]).sort_values("Stand-alone Sharpe", ascending=False)
print(f"\nThe {K_STAR} selected stocks (first 20 by stand-alone Sharpe ratio):")
display(stock_tab.head(20).round(2))
stock_tab.to_csv(f"{OUT_DIR}/qubo_selection_stocks.csv")

cmp_rows = {}
for name, w_ in {f"QUBO / SA (k = {K_STAR}, equal weights)": w_star, f"Equal weight, all {N} stocks": w_ew,
                 "Markowitz max-Sharpe": w_maxsh, "Markowitz min-variance": w_minvar}.items():
    r_, v_, s_, h_ = port_stats(w_)
    cmp_rows[name] = {"Exp. return (%)": r_ * 100, "Variance": v_, "Volatility (%)": s_ * 100, "Sharpe": h_,
                      "Objective": obj_w(w_), "No. of stocks": int((w_ > 1e-6).sum())}
CMP = pd.DataFrame(cmp_rows).T
print("\nFull-period risk / return (same mu, Sigma and lambda for every row):")
display(CMP.round(4))
CMP.to_csv(f"{OUT_DIR}/qubo_selection_comparison.csv")

fig, axes = plt.subplots(1, 2, figsize=(18, 7.5), gridspec_kw={"width_ratios": [1.25, 1]})
ax = axes[0]
ax.scatter(xs(np.diag(Sigma)), mu, s=10, color="lightgrey", edgecolor="grey", linewidths=0.3, label="Stocks", zorder=2)
ax.plot(xs(f_var), f_ret, color="red", lw=2.5, label=f"Efficient frontier (long-only, cap {MAX_W:.0%})", zorder=3)
qv = np.array([port_stats(SOL[k])[1] for k in K_LIST]); qr = np.array([port_stats(SOL[k])[0] for k in K_LIST])
ax.plot(xs(qv), qr, "k^--", lw=1, ms=7, label="QUBO solution for each k in K_LIST", zorder=4)
for k_, x_, y_ in zip(K_LIST, xs(qv), qr):
    ax.annotate(str(k_), (x_, y_), xytext=(4, 4), textcoords="offset points", fontsize=8)
r_s, v_s, _, _ = port_stats(w_star)
ax.scatter(xs(v_s), r_s, marker="*", s=600, color="gold", edgecolor="black", linewidths=1.5, zorder=6, label=f"QUBO choice: k = {K_STAR}")
ax.set_xlabel(XLABEL); ax.set_ylabel("Annualised expected return"); ax.grid(True, alpha=0.3); ax.legend(loc="lower right")
ax.set_title("Risk-return map: the QUBO selection", fontweight="bold")
ax = axes[1]
ax.plot(K_LIST, QK["Objective (QUBO/SA)"], color="#1f77b4", marker="o", label="Objective  lambda/2 w'Sigma w - mu'w  (lower = better)")
ax.axvline(K_STAR, color="gold", lw=3, alpha=0.8, label=f"chosen k = {K_STAR}")
ax.set_xlabel("k  (number of stocks selected)"); ax.set_ylabel("Objective of the QUBO solution", color="#1f77b4"); ax.grid(True, alpha=0.3)
ax2 = ax.twinx(); ax2.plot(K_LIST, QK["Sharpe"], color="#ff7f0e", marker="s", label="Sharpe ratio"); ax2.set_ylabel("Sharpe ratio", color="#ff7f0e")
ax2.spines["right"].set_visible(True)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels(); ax.legend(h1 + h2, l1 + l2, loc="best", fontsize=9)
ax.set_title("Quality of the QUBO solution for each k", fontweight="bold")
fig.suptitle(f"QUBO portfolio selection on {N} S&P 500 stocks, {returns.index[0].year}-{returns.index[-1].year}  (lambda = {LAMBDA:.3f})", fontsize=15, fontweight="bold")
plt.tight_layout(); savefig("qubo_selection_k_choice"); plt.show()

K = K_STAR            # number of stocks used by every solver from here on

### 6.4 The same QUBO sampled with D-Wave's classical simulated-annealing sampler

The QUBO of Section 5.1 for $k=k^\star$ is written as a `dimod.BinaryQuadraticModel` and sampled with the `SimulatedAnnealingSampler` of `dwave-samplers` (classical, runs on your computer, no account needed), with `NUM_READS` reads of `NUM_SWEEPS` sweeps (Section 4.1).

**What the next cell does**

* takes the penalty $P=P_{k^\star}$ of Section 5.3, builds $Q$, the constant and the `BinaryQuadraticModel`;
* samples the model, checks that the energy reported by the sampler equals $x^\top Qx+\text{const}$;
* reports the share of reads with exactly $k$ stocks and, for these reads, the **objective** (for a feasible read the QUBO energy *is* the Markowitz objective, because the penalty is zero), its best, mean and standard deviation, and the runtime;
* decodes the best portfolio the same way as for the own annealer (best of the 10 lowest-energy reads, repaired if needed);
* draws the histogram of the objectives of the feasible reads (with the own-annealer result) and the sampled portfolios in the risk-return plane.

In [ ]:
# SECTION 6.4 | D-Wave (classical SA sampler) version of the k-stock QUBO, k = K*
dimod = _ensure("dimod", "dimod")
_ensure("dwave.samplers", "dwave-samplers")
from dwave.samplers import SimulatedAnnealingSampler

P = float(P_of_k[K])                                           # penalty of Section 5.3 for this k
Q, CONST = build_qubo(K, P)
Qd = {(i, j): float(Q[i, j] * (1 if i == j else 2)) for i in range(N) for j in range(i, N)}   # x'Qx = sum_i Q_ii x_i + sum_{i<j} 2 Q_ij x_i x_j
bqm_qubo = dimod.BinaryQuadraticModel.from_qubo(Qd, offset=CONST)
print(f"k = {K} | N = {N} | lambda = {LAMBDA:.4f} | penalty P = {P:.5f}")

t0 = time.time()
ss_q = SimulatedAnnealingSampler().sample(bqm_qubo, num_reads=NUM_READS, num_sweeps=NUM_SWEEPS, seed=SEED)
t_dq = time.time() - t0
X_dq = np.asarray(ss_q.record.sample, dtype=float)                 # reads x N bits (variables are 0..N-1)
E_dq = np.asarray(ss_q.record.energy, dtype=float)
assert np.allclose(E_dq[:200], ((X_dq[:200] @ Q) * X_dq[:200]).sum(axis=1) + CONST), "sampler energy != x'Qx + const"
feas_dq = X_dq.sum(axis=1) == K
print(f"\n{NUM_READS} reads x {NUM_SWEEPS} sweeps in {t_dq:.1f} s | reads with exactly k stocks: {feas_dq.mean() * 100:.1f}%")
if feas_dq.any():
    obj_dq = E_dq[feas_dq]                                         # penalty = 0 for these reads: energy = Markowitz objective
    j_ = int(np.argmin(E_dq + np.where(feas_dq, 0, 1e9)))
    assert np.isclose(E_dq[j_], obj_w(X_dq[j_] / K), atol=1e-9)
    print(f"objective of the feasible reads: best {obj_dq.min():.6f} | mean {obj_dq.mean():.6f} | std {obj_dq.std():.6f}")
w_dq, info_dq = decode_select(X_dq, E_dq, dict(k=K), mu, Sigma, LAMBDA)
r_dq, v_dq, s_dq, h_dq = port_stats(w_dq)
print(f"best decoded portfolio: objective {obj_w(w_dq):.6f} | return {r_dq * 100:.2f}% | volatility {s_dq * 100:.2f}% | Sharpe {h_dq:.3f}")
print(f"own simulated annealer ({SA_CHAINS} chains x {SA_SWEEPS} sweeps) at k = {K}: objective {obj_w(w_star):.6f}")
FOUND = {f"QUBO + own SA ({SA_CHAINS} x {SA_SWEEPS})": obj_w(w_star), "QUBO + D-Wave SA sampler": obj_w(w_dq)}    # best objective found by each solver

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
ax = axes[0]
if feas_dq.any():
    ax.hist(obj_dq, bins=40, color="#2ca02c", alpha=0.85, label=f"feasible reads of the D-Wave sampler ({feas_dq.sum()} of {NUM_READS})")
ax.axvline(obj_w(w_star), color="red", ls="--", lw=2, label=f"own annealer best: {obj_w(w_star):.4f}")
ax.set_xlabel("objective of the read (QUBO energy)"); ax.set_ylabel("number of reads"); ax.grid(True, alpha=0.3); ax.legend()
ax.set_title(f"QUBO energies of the sampled bit strings (k = {K})", fontweight="bold")
ax = axes[1]
Wd = X_dq[feas_dq] / K
ax.scatter(xs(np.diag(Sigma)), mu, s=10, color="lightgrey", edgecolor="grey", linewidths=0.3, label="Stocks", zorder=2)
ax.plot(xs(f_var), f_ret, color="red", lw=2.5, label="Efficient frontier", zorder=3)
if len(Wd):
    ax.scatter(xs(((Wd @ Sigma) * Wd).sum(axis=1)), Wd @ mu, s=14, color="#1f77b4", alpha=0.5, label="sampled feasible portfolios", zorder=4)
ax.scatter(xs(v_dq), r_dq, marker="*", s=500, color="gold", edgecolor="black", zorder=6, label="best decoded portfolio")
ax.set_xlabel(XLABEL); ax.set_ylabel("Annualised expected return"); ax.grid(True, alpha=0.3); ax.legend(loc="lower right")
ax.set_title("Sampled portfolios in the risk-return plane", fontweight="bold")
plt.tight_layout(); savefig(f"dwave_qubo_k{K}"); plt.show()

## Section 7. Classical solver 2: MIQP (mixed-integer quadratic programming)

The MIQP chooses **which** $k$ stocks to hold and their **optimal (not necessarily equal) weights** at the same time:

$$\min_{w,z}\;\frac{\lambda}{2}\,w^\top\Sigma w-\mu^\top w\quad\text{s.t.}\quad \sum_i w_i=1,\;\; \sum_i z_i=k,\;\; W_{\min}z_i\le w_i\le z_i,\;\; z_i\in\{0,1\}.$$

$\mu$, $\Sigma$, $\lambda$ and $k=k^\star$ are the same as for the QUBO. The equal-weight portfolios of the QUBO are feasible points of this problem, so **the MIQP optimum is a lower bound for the optimum of the equal-weight QUBO**.

**Solvers.** Gurobi is tried first (it needs a full licence for this size: the licence that comes with `pip install gurobipy` is limited to a few hundred variables and is rejected). If it cannot be used, **SCIP** (free, `pyscipopt`) is used. For SCIP the dense quadratic form is rewritten with the eigen-decomposition $\Sigma=V\,\mathrm{diag}(\rho)\,V^\top$ ($\rho_i\ge0$) as $w^\top\Sigma w=\sum_i\rho_i u_i^2$ with $u=V^\top w$: a sum of squares of single variables, which is **exactly equal** to the original quadratic form (the notebook checks the objective with the original $\Sigma$). The epigraph variable $t\ge\frac\lambda2\sum_i\rho_iu_i^2-\mu^\top w$ is minimised.

**What the next cell does**

* sets `W_MIN = 0` (the model above) and solves the MIQP with a time limit `MIQP_TIME_LIMIT`;
* prints the solver, status, runtime, the objective computed with the original $\Sigma$, the solver's **lower bound** and the relative gap; the status tells you whether the solution is proven optimal or the time limit was reached (then the bound shows how far from optimal it can be);
* compares MIQP / equal-weight QUBO / equal weight of all stocks / maximum-Sharpe / minimum-variance and draws the risk-return map and the weights.

In [ ]:
# SECTION 7 | MIQP (Mixed-Integer Quadratic Programming) for the k-stock Markowitz problem, k = K*
#
#     minimise    lambda/2 * w'Sigma w  -  mu'w
#     subject to  sum(w) = 1 ,   sum(z) = k ,   W_MIN * z_i <= w_i <= z_i ,   z_i in {0,1}
#
# Gurobi (full licence) is tried first, then SCIP with the exact eigenvalue form of the quadratic term.

W_MIN = 0.0          # 0 -> exactly the model above. Set e.g. 0.01 to force every chosen stock to weigh >= 1%

ev, V = np.linalg.eigh(Sigma)
ev = np.clip(ev, 0, None)                                       # eigenvalues rho_i >= 0
Sigma_psd = (V * ev) @ V.T                                      # Sigma without tiny negative rounding eigenvalues (convex objective)

def solve_gurobi():
    gp = _ensure("gurobipy", "gurobipy")
    from gurobipy import GRB
    m = gp.Model("miqp_k_stocks"); m.Params.OutputFlag = 0; m.Params.TimeLimit = MIQP_TIME_LIMIT
    m.Params.MIPGap = 1e-9; m.Params.MIPGapAbs = 1e-12
    w = m.addMVar(N, lb=0.0, ub=1.0, name="w"); z = m.addMVar(N, vtype=GRB.BINARY, name="z")
    m.setObjective(0.5 * LAMBDA * (w @ Sigma_psd @ w) - mu @ w, GRB.MINIMIZE)
    m.addConstr(w.sum() == 1); m.addConstr(z.sum() == K); m.addConstr(w <= z); m.addConstr(w >= W_MIN * z)
    m.optimize()
    if m.SolCount == 0:
        raise RuntimeError(f"Gurobi status {m.Status}, no solution found")
    return w.X.copy(), z.X.copy(), f"status {m.Status} (2 = optimal, 9 = time limit)", float(m.ObjBound), float(m.MIPGap)

def solve_scip():
    sc = _ensure("pyscipopt", "pyscipopt")
    m = sc.Model("miqp_k_stocks"); m.hideOutput(); m.setParam("limits/time", float(MIQP_TIME_LIMIT))
    m.setParam("limits/gap", 1e-9); m.setParam("limits/absgap", 1e-12)
    w = [m.addVar(lb=0.0, ub=1.0, name=f"w{i}") for i in range(N)]
    z = [m.addVar(vtype="B", name=f"z{i}") for i in range(N)]
    u = [m.addVar(lb=None, name=f"u{i}") for i in range(N)]       # u = V'w
    t = m.addVar(lb=None, name="t")                                # epigraph variable
    m.setObjective(t, "minimize")
    for i in range(N):
        m.addCons(u[i] == sc.quicksum(float(V[j, i]) * w[j] for j in range(N)))
    m.addCons(t >= 0.5 * LAMBDA * sc.quicksum(float(ev[i]) * u[i] * u[i] for i in range(N)) - sc.quicksum(float(mu[i]) * w[i] for i in range(N)))
    m.addCons(sc.quicksum(w) == 1); m.addCons(sc.quicksum(z) == K)
    for i in range(N):
        m.addCons(w[i] <= z[i]); m.addCons(w[i] >= W_MIN * z[i])
    m.optimize()
    if m.getNSols() == 0:
        raise RuntimeError(f"SCIP status {m.getStatus()}, no solution found")
    return (np.array([m.getVal(v) for v in w]), np.array([m.getVal(v) for v in z]),
            f"status {m.getStatus()}", float(m.getDualbound()), float(m.getGap()))

SOLVER, t0 = None, time.time()
for name, fn in [("Gurobi", solve_gurobi), ("SCIP", solve_scip)]:
    try:
        w_raw, z_raw, status, MIQP_BOUND, MIQP_GAP = fn(); SOLVER = name; break
    except Exception as e:
        print(f"{name} could not be used: {str(e)[:150]}")
if SOLVER is None:
    raise RuntimeError("No MIQP solver worked. In a notebook cell run:  %pip install pyscipopt   then restart the kernel.")
t_miqp = time.time() - t0
w_miqp = np.where(w_raw < 1e-8, 0.0, w_raw); w_miqp /= w_miqp.sum()
z_miqp = np.flatnonzero(np.round(z_raw) > 0.5)
MIQP_OBJ = obj_w(w_miqp)                                         # objective with the ORIGINAL Sigma
print(f"MIQP solved with {SOLVER} in {t_miqp:.1f} s  |  {status}")
print(f"objective (original Sigma) {MIQP_OBJ:.6f} | lower bound of the solver {MIQP_BOUND:.6f} | relative gap (objective - bound)/|objective| = {(MIQP_OBJ - MIQP_BOUND) / abs(MIQP_OBJ):.2e}")
print(f"  (solver-reported gap {MIQP_GAP:.2e}; time limit {MIQP_TIME_LIMIT} s)")

held = np.flatnonzero(w_miqp > 1e-6)
print("\n" + "=" * 78)
print(f"MIQP RESULT (k = {K}): {len(held)} stocks with positive weight (z selects {len(z_miqp)}); largest weights:")
print(",  ".join(f"{tickers[i]} {w_miqp[i] * 100:.2f}%" for i in held[np.argsort(-w_miqp[held])][:15]) + (" ..." if len(held) > 15 else ""))
print("=" * 78)
tab = pd.DataFrame({"Annualised return (%)": mu * 100, "Volatility (%)": np.sqrt(np.diag(Sigma)) * 100, "MIQP weight (%)": w_miqp * 100}, index=tickers)
tab["Selected (z=1)"] = np.where(np.isin(np.arange(N), z_miqp), "YES", "-")
display(tab[tab["Selected (z=1)"] == "YES"].sort_values("MIQP weight (%)", ascending=False).head(20).round(3))
tab.to_csv(f"{OUT_DIR}/miqp_k{K}_weights.csv")

rows = {f"MIQP (k = {K}, optimal weights)": w_miqp, f"QUBO / SA (k = {K}, equal weights)": w_star,
        f"Equal weight, all {N} stocks": w_ew, "Markowitz max-Sharpe": w_maxsh, "Markowitz min-variance": w_minvar}
cmp_rows = {}
for name, w_ in rows.items():
    r_, v_, s_, h_ = port_stats(w_)
    cmp_rows[name] = {"Exp. return (%)": r_ * 100, "Variance": v_, "Volatility (%)": s_ * 100, "Sharpe": h_,
                      "Objective": obj_w(w_), "No. of stocks": int((w_ > 1e-6).sum())}
CMP2 = pd.DataFrame(cmp_rows).T
print("\nFull-period risk / return (same mu, Sigma and lambda for every row):")
display(CMP2.round(4))
CMP2.to_csv(f"{OUT_DIR}/miqp_k{K}_comparison.csv")
FOUND_MIQP = MIQP_OBJ

fig, axes = plt.subplots(1, 2, figsize=(18, 8), gridspec_kw={"width_ratios": [1.25, 1]})
ax = axes[0]
ax.scatter(xs(np.diag(Sigma)), mu, s=10, color="lightgrey", edgecolor="grey", linewidths=0.3, label="Stocks", zorder=2)
ax.plot(xs(f_var), f_ret, color="red", lw=2.5, label=f"Efficient frontier (long-only, cap {MAX_W:.0%})", zorder=3)
ax.scatter(xs(np.diag(Sigma)[held]), mu[held], s=40, color="#2ca02c", edgecolor="black", label="Stocks held by MIQP", zorder=4)
r_, v_, _, _ = port_stats(w_star)
ax.scatter(xs(v_), r_, marker="D", s=170, color="#1f77b4", edgecolor="black", zorder=5, label=f"QUBO / SA portfolio (k = {K}, equal weights)")
r_, v_, _, _ = port_stats(w_miqp)
ax.scatter(xs(v_), r_, marker="*", s=650, color="gold", edgecolor="black", linewidths=1.5, zorder=6, label=f"MIQP portfolio (k = {K}, free weights)")
ax.set_xlabel(XLABEL); ax.set_ylabel("Annualised expected return"); ax.grid(True, alpha=0.3); ax.legend(loc="lower right")
ax.set_title("Risk-return map: the MIQP portfolio", fontweight="bold")
ax = axes[1]
idx = np.array(sorted(set(held) | set(np.flatnonzero(w_star))))
x_ = np.arange(len(idx)); bw = 0.4
ax.bar(x_ - bw / 2, w_miqp[idx] * 100, bw, color="gold", edgecolor="black", label="MIQP weight")
ax.bar(x_ + bw / 2, w_star[idx] * 100, bw, color="#1f77b4", edgecolor="black", label="QUBO / SA equal weight")
ax.set_xticks(x_); ax.set_xticklabels([tickers[i] for i in idx], rotation=90, fontsize=6)
ax.set_ylabel("Portfolio weight (%)"); ax.grid(True, axis="y", alpha=0.3); ax.legend()
ax.set_title("Weights of the chosen stocks", fontweight="bold")
fig.suptitle(f"MIQP: best {K} of {N} US stocks, {returns.index[0].year}-{returns.index[-1].year}  (lambda = {LAMBDA:.3f}, solver {SOLVER})", fontsize=15, fontweight="bold")
plt.tight_layout(); savefig(f"miqp_k{K}_selection"); plt.show()

## Section 8. Ising spin-glass model

The QUBO of Section 5 is rewritten as an **Ising Hamiltonian** and solved. Everything that is used is the same as in the QUBO sections: same data, $\mu$, $\Sigma$, $\lambda$, $k=k^\star$ and $P=P_{k^\star}$. Any difference in the results below therefore comes only from the Ising formulation or from the solver.

### 8.1 QUBO to Ising (the mathematics) and validation

Substituting $x_i=(1+s_i)/2$, $s_i\in\{-1,+1\}$, into $H(x)=x^\top Qx+\text{const}$ gives

$$H(s)=\sum_{i<j}J_{ij}\,s_is_j+\sum_i h_i\,s_i+C,\qquad J_{ij}=\tfrac12 Q_{ij},\quad h_i=\tfrac12\sum_j Q_{ij},\quad C=\tfrac14\sum_{ij}Q_{ij}+\tfrac14\sum_i Q_{ii}+Pk^2 .$$

Bit 1 (stock selected) $\leftrightarrow$ spin $+1$. The constraint "exactly $k$ stocks" becomes a **fixed magnetisation** $\sum_i s_i=2k-N$. With $Q$ of Section 5.1,

$$J_{ij}=\frac P2+\frac{\lambda}{4k^2}\Sigma_{ij},\qquad h_i=\frac{P(N-2k)}{2}-\frac{\mu_i}{2k}+\frac{\lambda}{4k^2}\sum_j\Sigma_{ij}$$

(checked numerically in the next cell).

**What the next cell does**

* converts the QUBO with the hand-coded formulas (`qubo_to_ising_manual`) and prints the number of positive / negative couplings and fields, their ranges, the split into penalty / return / covariance parts, the number of **frustrated triangles** (triples with $J_{ij}J_{jl}J_{il}>0$; counted exactly with the identity $\sum_{i<j<l}\operatorname{sgn}J_{ij}\operatorname{sgn}J_{jl}\operatorname{sgn}J_{il}=\operatorname{tr}(S^3)/6$ for the sign matrix $S$, because looping over all $\binom{N}{3}$ triples is too slow) and the fixed magnetisation;
* compares with the `dimod` conversion of the QUBO of Section 6.4 (`bqm.to_ising()`): the two must agree to rounding error;
* **validates** the Ising energy on random states, because all $2^N$ states cannot be enumerated: on 200 000 random spin states $E_{\text{Ising}}(s)=E_{\text{QUBO}}(x)$ with $x=(1+s)/2$, and on 5 000 random states with exactly $k$ stocks $E_{\text{Ising}}$ equals the Markowitz objective of the equal-weight portfolio. An `assert` stops the notebook if a check fails.

In [ ]:
# SECTION 8.1 | QUBO (k = K*) -> Ising: hand-coded conversion, comparison with dimod, validation on random states
M_FIXED = 2 * K - N                         # fixed magnetisation: sum(s) = 2k - N

def build_Q(Pm):
    """QUBO matrix and constant of Section 5.1 for k = K*:  H(x) = x'Qx + CONST."""
    return build_qubo(K, Pm)

def qubo_to_ising_manual(Q, const):
    J = np.triu(Q / 2.0, 1)                    # J_ij = Q_ij / 2   (i < j)
    h = 0.5 * Q.sum(axis=1)                    # h_i  = 1/2 * sum_j Q_ij
    C = 0.25 * Q.sum() + 0.25 * np.trace(Q) + const
    return J, h, C

def ising_energy(S, J=None, h=None, C=None):
    """S: (m, N) array of +-1 spins -> energies (m,).   H = sum_{i<j} J_ij s_i s_j + sum_i h_i s_i + C"""
    J = _J0 if J is None else J; h = _H0 if h is None else h; C = _C0 if C is None else C
    return S @ h + ((S @ J) * S).sum(axis=1) + C

J, h, C = qubo_to_ising_manual(Q, CONST)
_J0, _H0, _C0 = J, h, C                       # defaults of ising_energy
iu = np.triu_indices(N, 1)
Jp = J[iu]
print(f"{len(iu[0]):,} spin-spin couplings (fully connected, N = {N}) | k = {K} | P = {P:.5f} | lambda = {LAMBDA:.4f}")
print(f"J_ij : {(Jp > 0).sum():,} positive (antiferromagnetic), {(Jp < 0).sum():,} negative (ferromagnetic) | "
      f"range [{Jp.min():.6f}, {Jp.max():.6f}] | mean {Jp.mean():.6f} | std {Jp.std():.6f}")
J_pen, J_cov = P / 2, LAMBDA / (4 * K ** 2) * Sigma
assert np.allclose(Jp, J_pen + J_cov[iu])
print(f"  J_ij = P/2 + lambda/(4k^2) Sigma_ij (verified): penalty part P/2 = {J_pen:.6f} for every pair "
      f"({J_pen / Jp.mean() * 100:.0f}% of the mean coupling); covariance part in [{J_cov[iu].min():.6f}, {J_cov[iu].max():.6f}]")
print(f"h_i  : {(h > 0).sum()} positive, {(h < 0).sum()} negative | range [{h.min():.6f}, {h.max():.6f}]")
h_pen, h_ret, h_cov = P * (N - 2 * K) / 2, -mu / (2 * K), LAMBDA / (4 * K ** 2) * Sigma.sum(axis=1)
assert np.allclose(h, h_pen + h_ret + h_cov)
print(f"  h_i = P(N-2k)/2 - mu_i/(2k) + lambda/(4k^2) sum_j Sigma_ij (verified): penalty part {h_pen:.6f} for every spin | "
      f"return part in [{h_ret.min():.6f}, {h_ret.max():.6f}] | covariance part in [{h_cov.min():.6f}, {h_cov.max():.6f}]")
Js = J + J.T
n_tri = comb(N, 3)
if (Jp != 0).all():                                                  # sign product of every triple is +1 or -1
    Sg = np.sign(Js)
    frustrated = int(round((n_tri + np.trace(Sg @ Sg @ Sg) / 6) / 2))   # #(product > 0) = (n_tri + sum of products) / 2
    print(f"frustrated triangles (J_ij J_jl J_il > 0): {frustrated:,} of {n_tri:,}")
else:
    frustrated = None
    print("some couplings are exactly zero: the number of frustrated triangles is not computed")
print(f"constant C = {C:.6f} | fixed magnetisation: sum(s) = 2k - N = {M_FIXED}  ({K} spins up, {N - K} down)")

# comparison with dimod's own conversion of the QUBO of Section 6.4
h_d, J_d, off_d = bqm_qubo.to_ising()
hd = np.array([h_d[i] for i in range(N)]); Jd = np.zeros((N, N))
for (a, b), v in J_d.items():
    Jd[min(a, b), max(a, b)] += v
print(f"\nhand-coded vs dimod to_ising(): max |h diff| = {np.abs(h - hd).max():.2e} | max |J diff| = {np.abs(J - Jd).max():.2e} | |C diff| = {abs(C - off_d):.2e}")
assert np.allclose(h, hd, atol=1e-10) and np.allclose(J, Jd, atol=1e-10) and np.isclose(C, off_d, atol=1e-9)

# validation on random states (all 2^N states cannot be enumerated)
rng_v = np.random.default_rng(SEED)
max_rel = 0.0
for _ in range(20):
    S_ = rng_v.choice([-1.0, 1.0], size=(10000, N)); X_ = (S_ + 1) / 2
    e_i = ising_energy(S_); e_q = ((X_ @ Q) * X_).sum(axis=1) + CONST
    max_rel = max(max_rel, np.max(np.abs(e_i - e_q) / np.maximum(1.0, np.abs(e_q))))
print(f"validation 1 - 200,000 random spin states: max relative |E_Ising - E_QUBO| = {max_rel:.1e}")
assert max_rel < 1e-10
S_f = -np.ones((5000, N))
for r_ in range(5000):
    S_f[r_, rng_v.choice(N, K, replace=False)] = 1.0
e_f = ising_energy(S_f); f_f = np.array([obj_w(((s_ + 1) / 2) / K) for s_ in S_f])
max_diff = float(np.abs(e_f - f_f).max())
print(f"validation 2 - 5,000 random states with exactly {K} stocks: max |E_Ising - Markowitz objective| = {max_diff:.1e}")
assert max_diff < 1e-9

### 8.2 Ising model for the D-Wave sampler

**What the next cell does**: builds the `dimod` `BinaryQuadraticModel` of the **spin** model (`from_ising`, with $h$, $J$ and the offset $C$) that is sampled in the next sections, and checks that it has the same energies as `ising_energy` on random spin states.

In [ ]:
# SECTION 8.2 | spin (Ising) BQM for the D-Wave sampler
def make_ising_bqm(Jm, hm, Cm):
    return dimod.BinaryQuadraticModel.from_ising({i: float(hm[i]) for i in range(N)},
                                                 {(int(i), int(j)): float(Jm[i, j]) for i, j in zip(*iu)}, float(Cm))
bqm_ising = make_ising_bqm(J, h, C)
S_t = rng_v.choice([-1.0, 1.0], size=(50, N))
e_bqm = bqm_ising.energies((S_t, list(range(N))))
print(f"BQM vartype {bqm_ising.vartype.name} | {bqm_ising.num_variables} spins | {bqm_ising.num_interactions:,} couplings | "
      f"max |BQM energy - ising_energy| on 50 random states = {np.abs(e_bqm - ising_energy(S_t)).max():.1e}")
assert np.allclose(e_bqm, ising_energy(S_t), atol=1e-9)

### 8.3 Ising simulated annealing with the D-Wave sampler (main run)

The same budget and seed as the QUBO run of Section 6.4 (`NUM_READS` reads x `NUM_SWEEPS` sweeps, seed `SEED`), now on the **spin model**. A read is **feasible** if it has exactly $k$ spins up. For a feasible read the Ising energy is the Markowitz objective. No exact optimum is available, so the result is compared with the QUBO run and (in Section 9) with the best known value.

**What the next cell does**: samples the Ising model, checks that the energy reported by the sampler equals `ising_energy` of the returned spins, converts the spins back to bits ($x=(1+s)/2$), prints the share of feasible reads and the best / mean / standard deviation of their objective, the runtime, and decodes the best portfolio the same way as for the QUBO.

In [ ]:
# SECTION 8.3 | Ising simulated annealing with the D-Wave sampler (SPIN model), same budget and seed as the QUBO run of Section 6.4
t0 = time.time()
ss_i = SimulatedAnnealingSampler().sample(bqm_ising, num_reads=NUM_READS, num_sweeps=NUM_SWEEPS, seed=SEED)
t_di = time.time() - t0
S_di = np.asarray(ss_i.record.sample, dtype=float)                   # reads x N spins (+-1)
E_di = np.asarray(ss_i.record.energy, dtype=float)
assert np.allclose(E_di[:200], ising_energy(S_di[:200]), atol=1e-9), "sampler energy != ising_energy"
X_di = (S_di + 1) / 2
feas_di = S_di.sum(axis=1) == M_FIXED
print(f"Ising: {NUM_READS} reads x {NUM_SWEEPS} sweeps in {t_di:.1f} s | reads with exactly k stocks: {feas_di.mean() * 100:.1f}%")
if feas_di.any():
    obj_di = E_di[feas_di]
    j_ = int(np.argmin(E_di + np.where(feas_di, 0, 1e9)))
    assert np.isclose(E_di[j_], obj_w(X_di[j_] / K), atol=1e-9)
    print(f"objective of the feasible reads: best {obj_di.min():.6f} | mean {obj_di.mean():.6f} | std {obj_di.std():.6f}")
print(f"QUBO (Section 6.4)  : feasible reads {feas_dq.mean() * 100:.1f}% | best {obj_dq.min():.6f} | mean {obj_dq.mean():.6f} | {t_dq:.1f} s")
w_ising_dw, info_di = decode_select(X_di, E_di, dict(k=K), mu, Sigma, LAMBDA)
r_i, v_i, s_i, sh_i = port_stats(w_ising_dw)
print(f"\nbest decoded Ising portfolio: objective {obj_w(w_ising_dw):.6f} | return {r_i * 100:.2f}% | volatility {s_i * 100:.2f}% | Sharpe {sh_i:.3f}")
print(f"same stocks as the QUBO D-Wave portfolio: {set(np.flatnonzero(w_ising_dw)) == set(np.flatnonzero(w_dq))} | "
      f"stocks in common: {len(set(np.flatnonzero(w_ising_dw)) & set(np.flatnonzero(w_dq)))} of {K}")
FOUND["Ising + D-Wave SA sampler"] = obj_w(w_ising_dw)

### 8.4 Own Ising simulated annealer (single-spin flips and two-spin moves)

Besides the library sampler, the Ising model is annealed with the **own simulated annealer written for spins** (the one of the 20-stock notebook):

* local field $f_i=h_i+\sum_j J_{ij}s_j$; flipping $s_i\to-s_i$ costs $\Delta E=-2s_if_i$;
* `n_chains` chains in parallel, temperature falling geometrically over `n_sweeps` sweeps (start $=0.1\times$ the median $|\Delta E|$ of random states, end $=10^{-4}\times$ start);
* with `pair_moves=True`, after the single flips of every sweep, $N$ **two-spin moves** are proposed (spins $i$ and $k$ flipped together; with one up and one down this swaps a stock in for a stock out and keeps $\sum_i s_i$ fixed), with cost $\Delta E=-2s_if_i-2s_kf_k+4J_{ik}s_is_k$. These are the same moves as the two-bit moves of the QUBO annealer of Section 6.1.

**What the next cell does**: defines `ising_sa`, runs it once with single flips and once with single flips plus two-spin moves (`SA_CHAINS` x `SA_SWEEPS` each, the budget of the own QUBO annealer), prints the runtime, the share of feasible reads and the best / mean objective of the feasible reads, and decodes the best portfolio of each run with the same `decode_select` and `obj_w` as for the QUBO.

In [ ]:
# SECTION 8.4 | Own Ising simulated annealer: Metropolis single-spin flips (+ optional two-spin moves) on h and J
#   local field  f_i = h_i + sum_j J_ij s_j     flip s_i -> -s_i costs   dE = -2 s_i f_i
#   two-spin move cost:  dE = -2 s_i f_i - 2 s_k f_k + 4 J_ik s_i s_k
def ising_sa(J, h, C, n_chains=128, n_sweeps=150, rng=None, t_scale=0.1, pair_moves=False):
    """Returns final spin states (chains x n) and their energies.  J is the upper-triangular coupling matrix."""
    rng = rng if rng is not None else np.random.default_rng(SEED)
    Jf = J + J.T                                                    # full symmetric couplings (zero diagonal)
    n = len(h)
    S = rng.choice([-1.0, 1.0], size=(n_chains, n))
    F = S @ Jf + h                                                  # local fields of every chain
    t_hi = t_scale * np.median(np.abs(-2 * S * F))
    temps = np.geomspace(t_hi, t_hi * 1e-4, n_sweeps)
    for T in temps:
        U = rng.random((n, n_chains))
        for j, i in enumerate(rng.permutation(n)):
            dE = -2 * S[:, i] * F[:, i]
            acc = (dE <= 0) | (U[j] < np.exp(-np.maximum(dE, 0) / T))
            if acc.any():
                m = -2 * S[:, i] * acc                              # change of s_i where the flip is accepted
                S[:, i] += m
                F += m[:, None] * Jf[i]
        if pair_moves:
            U2 = rng.random((n, n_chains))
            pairs = np.array([rng.choice(n, 2, replace=False) for _ in range(n)])
            for j, (i, k) in enumerate(pairs):
                dE = -2 * S[:, i] * F[:, i] - 2 * S[:, k] * F[:, k] + 4 * Jf[i, k] * S[:, i] * S[:, k]
                acc = (dE <= 0) | (U2[j] < np.exp(-np.maximum(dE, 0) / T))
                if acc.any():
                    mi, mk = -2 * S[:, i] * acc, -2 * S[:, k] * acc
                    S[:, i] += mi; S[:, k] += mk
                    F += mi[:, None] * Jf[i] + mk[:, None] * Jf[k]
    return S, ising_energy(S, J, h, C)

# one run of each kind with the budget of the own QUBO annealer: single-spin flips, and single flips + two-spin moves
RUNS = {}
def decode_spins(S):
    return (1 + S) / 2
q_k = dict(k=K)
for label, pm in [("Ising SA (own code), single flips", False), ("Ising SA (own code) + two-spin moves", True)]:
    t0 = time.time()
    S_run, E_run = ising_sa(J, h, C, SA_CHAINS, SA_SWEEPS, rng=np.random.default_rng(SEED), pair_moves=pm)
    dt = time.time() - t0
    feas = S_run.sum(axis=1) == M_FIXED
    w_run, _ = decode_select(decode_spins(S_run), E_run, q_k, mu, Sigma, LAMBDA)
    RUNS[label] = (S_run, E_run, dt, w_run)
    FOUND[label + f" ({SA_CHAINS} x {SA_SWEEPS})"] = obj_w(w_run)
    msg = (f"objective of the feasible reads: best {E_run[feas].min():.6f} | mean {E_run[feas].mean():.6f}" if feas.any() else "no feasible read")
    print(f"{label:<42}: {dt:6.1f} s | feasible reads {feas.mean() * 100:5.1f}% | {msg} | decoded portfolio objective {obj_w(w_run):.6f}")

### 8.5 Repeated experiments with the D-Wave sampler: Ising versus QUBO, sweeps and penalty

Each stochastic solver is repeated over `N_SEEDS` seeds with `EXP_READS` reads (smaller than the main run because one 1000-read run takes about a minute at this size). Metrics per seed: share of reads with exactly $k$ stocks, the best and the mean objective of the feasible reads, runtime. The QUBO model (D-Wave sampler) is run with the same seeds as a like-for-like reference.

**What the next cell does**

* **8.5a** headline comparison: Ising SA versus QUBO SA, `EXP_READS` reads x `EXP_SWEEPS` sweeps, `N_SEEDS` seeds;
* **8.5b** sensitivity of the Ising model to the **number of sweeps** (10, 50, 150, `EXP_SWEEPS`);
* **8.5c** sensitivity of the Ising model to the **penalty** $P$ (0.5x, 1x, 2x of the Monte-Carlo value) with `EXP_SWEEPS` sweeps.

Because there is no exact optimum, the table gives the best objective found, not a hit rate; the comparison with the best known value is made in Section 9. This is the slowest part of the notebook after the main runs.

In [ ]:
# SECTION 8.5 | repeated experiments with the D-Wave sampler (N_SEEDS seeds)
SEEDS = list(range(1, N_SEEDS + 1))
EXP_E = {}                                          # per method: list (one entry per seed) of the objectives of the feasible reads

def run_dwave(bqm, kind, reads, sweeps, seed):
    """One sampler run -> (feasible-read objectives, share of feasible reads, runtime). kind: 'ising' (spins) or 'qubo' (bits)."""
    t0 = time.time()
    ss = SimulatedAnnealingSampler().sample(bqm, num_reads=reads, num_sweeps=sweeps, seed=seed)
    dt = time.time() - t0
    Z = np.asarray(ss.record.sample, dtype=float); E = np.asarray(ss.record.energy, dtype=float)
    ones = (Z + 1) / 2 if kind == "ising" else Z
    feas = ones.sum(axis=1) == K
    return E[feas], float(feas.mean() * 100), dt

def summarise(label, runs):
    EXP_E[label] = [r[0] for r in runs]
    best = np.array([r[0].min() if len(r[0]) else np.nan for r in runs])
    mean = np.array([r[0].mean() if len(r[0]) else np.nan for r in runs])
    return {"Best objective (mean over seeds)": np.nanmean(best), "Best objective (best seed)": np.nanmin(best),
            "Best objective (worst seed)": np.nanmax(best), "Mean objective of feasible reads": np.nanmean(mean),
            "Feasible reads (%)": np.mean([r[1] for r in runs]), "Runtime per run (s)": np.mean([r[2] for r in runs])}

# 8.5a  Ising vs QUBO, same reads / sweeps / seeds
t_exp = time.time()
runs_i = [run_dwave(bqm_ising, "ising", EXP_READS, EXP_SWEEPS, s_) for s_ in SEEDS]
runs_q = [run_dwave(bqm_qubo, "qubo", EXP_READS, EXP_SWEEPS, s_) for s_ in SEEDS]
tab_main = pd.DataFrame({f"Ising SA (D-Wave sampler, {EXP_READS} x {EXP_SWEEPS})": summarise("Ising D-Wave", runs_i),
                         f"QUBO SA (D-Wave sampler, {EXP_READS} x {EXP_SWEEPS})": summarise("QUBO D-Wave", runs_q)}).T
print(f"8.5a  Ising vs QUBO, {EXP_READS} reads x {EXP_SWEEPS} sweeps, {N_SEEDS} seeds ({time.time() - t_exp:.0f} s):")
display(tab_main.round(6))

# 8.5b  number of sweeps (Ising)
SWEEPS = [10, 50, 150, EXP_SWEEPS]
rows_sw = {}
for sw_ in SWEEPS:
    rr = [run_dwave(bqm_ising, "ising", EXP_READS, sw_, s_) for s_ in SEEDS]
    rows_sw[sw_] = summarise(f"Ising D-Wave, {sw_} sweeps", rr)
tab_sweeps = pd.DataFrame(rows_sw).T
tab_sweeps.index.name = "sweeps"
print(f"\n8.5b  Ising SA versus the number of sweeps ({EXP_READS} reads, {N_SEEDS} seeds):")
display(tab_sweeps[["Best objective (mean over seeds)", "Mean objective of feasible reads", "Feasible reads (%)", "Runtime per run (s)"]].round(6))

# 8.5c  penalty P (Ising)
rows_P = {}
for mult in [0.5, 1.0, 2.0]:
    Qm, Cm = build_Q(mult * P); Jm, hm, Cmm = qubo_to_ising_manual(Qm, Cm)
    bqm_m = make_ising_bqm(Jm, hm, Cmm)
    rr = [run_dwave(bqm_m, "ising", EXP_READS, EXP_SWEEPS, s_) for s_ in SEEDS]
    rows_P[mult] = {"P": mult * P, **summarise(f"Ising D-Wave, P x {mult}", rr)}
tab_P = pd.DataFrame(rows_P).T
tab_P.index.name = "P multiplier"
print(f"\n8.5c  Ising SA versus the penalty ({EXP_READS} reads x {EXP_SWEEPS} sweeps, {N_SEEDS} seeds):")
display(tab_P[["P", "Best objective (mean over seeds)", "Mean objective of feasible reads", "Feasible reads (%)"]].round(6))
for nm, tb in [("main", tab_main), ("sweeps", tab_sweeps), ("penalty", tab_P)]:
    tb.to_csv(f"{OUT_DIR}/ising_dwave_experiments_{nm}.csv")

### 8.6 Repeated experiments with the own annealers

The same kind of experiment for the **own** annealers of Section 6.1 (QUBO) and Section 8.4 (Ising): `N_SEEDS` seeds (`SEED`, `SEED+1`, …), `SA_CHAINS` chains x `SA_SWEEPS` sweeps, the same metrics as in Section 8.5. The three annealers are: Ising with single flips, Ising with single flips plus two-spin moves, and the QUBO annealer of Section 6.1 (single plus two-bit moves).

In [ ]:
# SECTION 8.6 | repeated experiments with the own annealers (N_SEEDS seeds)
def own_run(fn, seed):
    t0 = time.time(); S, E = fn(np.random.default_rng(seed)); dt = time.time() - t0
    ones = (S + 1) / 2
    feas = ones.sum(axis=1) == K
    return E[feas], float(feas.mean() * 100), dt

def qubo_sa_as_spins(r):                              # QUBO annealer of Section 6.1 (bits) -> spins, so the same code applies
    X_, E_, _ = simulated_annealing(Q, SA_CHAINS, SA_SWEEPS, rng=r)
    return 2 * X_ - 1, E_ + CONST

t_exp = time.time()
own = {f"Ising SA (own code), single flips, {SA_CHAINS} x {SA_SWEEPS}": lambda r: ising_sa(J, h, C, SA_CHAINS, SA_SWEEPS, rng=r),
       f"Ising SA (own code) + two-spin moves, {SA_CHAINS} x {SA_SWEEPS}": lambda r: ising_sa(J, h, C, SA_CHAINS, SA_SWEEPS, rng=r, pair_moves=True),
       f"QUBO SA (own code of Section 6.1), {SA_CHAINS} x {SA_SWEEPS}": qubo_sa_as_spins}
rows_own = {}
for label, fn in own.items():
    runs = [own_run(fn, SEED + s_) for s_ in range(N_SEEDS)]
    rows_own[label] = summarise(label, runs)
tab_own = pd.DataFrame(rows_own).T
print(f"{N_SEEDS} seeds per solver ({time.time() - t_exp:.0f} s):")
display(tab_own.round(6))
tab_own.to_csv(f"{OUT_DIR}/ising_own_annealers_experiments.csv")

### 8.7 Charts

1. **Risk-return map**: all stocks, the efficient frontier, the MIQP portfolio and the best QUBO and Ising portfolios.
2. **Objectives of the feasible reads** (D-Wave sampler, main run): Ising versus QUBO, with the MIQP value.
3. **Sensitivity**: best objective and share of feasible reads versus the number of sweeps and versus the penalty (D-Wave sampler experiments).
4. **Spin-glass diagnostics**: heat map of the coupling matrix $J$ (stocks in the order of the table) and the histograms of $J_{ij}$ and $h_i$.

In [ ]:
# SECTION 8.7 | chart 1: risk-return map (QUBO / Ising / MIQP vs the efficient frontier)
fig, ax = plt.subplots(figsize=(13, 8))
ax.scatter(xs(np.diag(Sigma)), mu, s=10, color="lightgrey", edgecolor="grey", linewidths=0.3, label="Stocks", zorder=2)
ax.plot(xs(f_var), f_ret, color="red", lw=2.5, label=f"Efficient frontier (long-only, cap {MAX_W:.0%})", zorder=3)
for w_, mk, sz, col, lab in [(w_star, "^", 200, "#9467bd", f"QUBO + own SA (k = {K})"), (w_dq, "s", 150, "#1f77b4", "QUBO + D-Wave SA sampler"),
                             (w_ising_dw, "o", 130, "#ff7f0e", "Ising + D-Wave SA sampler"), (w_miqp, "D", 190, "#2ca02c", f"MIQP ({SOLVER}, free weights)")]:
    r_, v_, _, _ = port_stats(w_)
    ax.scatter(xs(v_), r_, marker=mk, s=sz, color=col, edgecolor="black", linewidths=1.2, zorder=6, label=lab)
ax.set_xlabel(XLABEL); ax.set_ylabel("Annualised expected return"); ax.grid(True, alpha=0.3); ax.legend(loc="lower right")
ax.set_title(f"Ising, QUBO and MIQP portfolios on the risk-return plane (k = {K}, lambda = {LAMBDA:.3f})", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("ising_risk_return_map"); plt.show()

# chart 2: objectives of the feasible reads, Ising vs QUBO (D-Wave sampler, main run)
fig, ax = plt.subplots(figsize=(12, 6.5))
allv = np.concatenate([obj_di, obj_dq]); bins = np.linspace(allv.min(), np.percentile(allv, 99), 45)
ax.hist(obj_dq, bins=bins, color="#1f77b4", alpha=0.6, label=f"QUBO SA: {len(obj_dq)} feasible reads")
ax.hist(obj_di, bins=bins, color="#ff7f0e", alpha=0.6, label=f"Ising SA: {len(obj_di)} feasible reads")
ax.axvline(MIQP_OBJ, color="#2ca02c", lw=2, ls="--", label=f"MIQP objective (free weights): {MIQP_OBJ:.4f}")
ax.axvline(MIQP_BOUND, color="black", lw=1.5, ls=":", label=f"MIQP lower bound: {MIQP_BOUND:.4f}")
ax.set_xlabel("objective of the read (Markowitz objective of the equal-weight portfolio)"); ax.set_ylabel("number of reads"); ax.grid(True, alpha=0.3); ax.legend()
ax.set_title(f"Objectives of the feasible reads, D-Wave sampler {NUM_READS} x {NUM_SWEEPS} (k = {K})", fontsize=13, fontweight="bold")
plt.tight_layout(); savefig("ising_vs_qubo_objective_histogram"); plt.show()

# chart 3: sensitivity to the number of sweeps and to the penalty
fig, axes = plt.subplots(1, 2, figsize=(17, 6.5))
ax = axes[0]
ax.plot(tab_sweeps.index, tab_sweeps["Best objective (mean over seeds)"], "o-", color="#1f77b4", lw=2, label="best objective (mean over seeds)")
ax.set_xscale("log"); ax.set_xticks(tab_sweeps.index); ax.set_xticklabels(tab_sweeps.index)
ax.set_xlabel("number of sweeps"); ax.set_ylabel("best objective of the feasible reads", color="#1f77b4"); ax.grid(True, alpha=0.3)
ax2 = ax.twinx(); ax2.plot(tab_sweeps.index, tab_sweeps["Feasible reads (%)"], "s--", color="#d62728", label="feasible reads (%)"); ax2.set_ylabel("feasible reads (%)", color="#d62728"); ax2.spines["right"].set_visible(True)
ax.set_title("Ising SA versus the number of sweeps", fontweight="bold")
ax = axes[1]
ax.plot(tab_P.index.astype(float), tab_P["Best objective (mean over seeds)"], "o-", color="#1f77b4", lw=2)
ax.set_xlabel("penalty multiplier (x P)"); ax.set_ylabel("best objective of the feasible reads", color="#1f77b4"); ax.set_xscale("log"); ax.set_xticks([0.5, 1, 2]); ax.set_xticklabels(["0.5", "1", "2"]); ax.grid(True, alpha=0.3)
ax2 = ax.twinx(); ax2.plot(tab_P.index.astype(float), tab_P["Feasible reads (%)"], "s--", color="#d62728"); ax2.set_ylabel("feasible reads (%)", color="#d62728"); ax2.spines["right"].set_visible(True)
ax.set_title("Ising SA versus the penalty P", fontweight="bold")
plt.tight_layout(); savefig("ising_sensitivity_sweeps_penalty"); plt.show()

# chart 4: spin-glass diagnostics
fig, axes = plt.subplots(1, 3, figsize=(20, 6.5), gridspec_kw={"width_ratios": [1.2, 1, 1]})
ax = axes[0]
im = ax.imshow(np.ma.masked_where(np.eye(N, dtype=bool), Js), cmap="viridis", vmin=Jp.min(), vmax=Jp.max())
plt.colorbar(im, ax=ax, label="J_ij"); ax.set_xlabel("stock index"); ax.set_ylabel("stock index")
ax.set_title("Coupling matrix J (penalty + covariance)", fontweight="bold")
ax = axes[1]; ax.hist(Jp, bins=60, color="#9467bd", edgecolor="white"); ax.axvline(J_pen, color="red", ls="--", label=f"penalty part P/2 = {J_pen:.5f}")
ax.set_xlabel("J_ij"); ax.set_ylabel("number of pairs"); ax.legend(); ax.set_title("Distribution of the couplings", fontweight="bold")
ax = axes[2]; ax.hist(h, bins=60, color="#17becf", edgecolor="white"); ax.axvline(h_pen, color="red", ls="--", label=f"penalty part = {h_pen:.4f}")
ax.set_xlabel("h_i"); ax.set_ylabel("number of spins"); ax.legend(); ax.set_title("Distribution of the local fields", fontweight="bold")
plt.tight_layout(); savefig("ising_J_h_diagnostics"); plt.show()

## Section 9. Comparison table and conclusions (objective 5)

### 9.1 Comparison table: simulated annealing and MIQP versus the Ising model (about 500 US stocks)

Every row is computed in this notebook from the same $\mu$, $\Sigma$, $\lambda$ and $k$, and scored with the same functions (`port_stats`, `obj_w`). Rows beginning with *Ising* are the Ising results (highlighted).

**There is no exact optimum for this problem size.** The table therefore shows, for every method, its objective, the **gap to the best known** objective (`F_BEST` = the lowest objective found by any equal-weight solver in this notebook, in the main runs and in all repeated experiments; it is the best value found, **not** a proven optimum) and the **gap to the MIQP lower bound**. The MIQP lower bound is valid for the equal-weight problem, because every equal-weight portfolio is a feasible point of the MIQP.

**What the next cell does**

* collects the portfolio of each method and the lowest objective found in the repeated experiments;
* computes `F_BEST` and, for the repeated experiments, the share of feasible reads whose objective is within $10^{-9}$ of `F_BEST`;
* prints the comparison table, saves it, and shows the solver statistics of the repeated experiments.

In [ ]:
# SECTION 9.1 | COMPARISON TABLE (about 500 US stocks): QUBO + simulated annealing, MIQP and Ising
def row(method, w):
    r_, v_, s_, h_ = port_stats(w)
    return {"Method": method, "Stocks held": int((w > 1e-6).sum()), "Return %": r_ * 100, "Vol %": s_ * 100, "Sharpe": h_, "Objective": obj_w(w)}

methods = [
    (f"MIQP ({SOLVER}, k = {K}, free weights)", w_miqp),
    (f"QUBO + own SA ({SA_CHAINS} x {SA_SWEEPS})", w_star),
    (f"QUBO + D-Wave SA sampler ({NUM_READS} x {NUM_SWEEPS})", w_dq),
    (f"Ising + D-Wave SA sampler ({NUM_READS} x {NUM_SWEEPS})", w_ising_dw),
    (f"Ising + own SA, single flips ({SA_CHAINS} x {SA_SWEEPS})", RUNS["Ising SA (own code), single flips"][3]),
    (f"Ising + own SA with two-spin moves ({SA_CHAINS} x {SA_SWEEPS})", RUNS["Ising SA (own code) + two-spin moves"][3]),
]
# lowest objective found by any equal-weight solver (main runs + every repeated experiment)
cand = [obj_w(w_) for m_, w_ in methods[1:]]
for label, lst in EXP_E.items():
    cand += [e.min() for e in lst if len(e)]
F_BEST = float(min(cand))
tab_rows = [row(m_, w_) for m_, w_ in methods]
cmp_tab = pd.DataFrame(tab_rows).set_index("Method")
cmp_tab["Gap to best known"] = cmp_tab["Objective"] - F_BEST
cmp_tab["Gap to MIQP bound"] = cmp_tab["Objective"] - MIQP_BOUND
cmp_tab.loc["Ising + QPU (quantum annealer)"] = [np.nan] * cmp_tab.shape[1]
cmp_tab = cmp_tab.astype(object); cmp_tab.loc["Ising + QPU (quantum annealer)", "Stocks held"] = "not run"
cmp_tab.to_csv(f"{OUT_DIR}/comparison_table_500stocks.csv")
def hl(r_):
    return ["background-color: #fff2a8" if r_.name.startswith("Ising") else "" for _ in r_]
print(f"best known equal-weight objective F_BEST = {F_BEST:.6f} | MIQP objective {MIQP_OBJ:.6f} | MIQP lower bound {MIQP_BOUND:.6f} | C({N},{K}) = {comb(N, K):.3e} portfolios (no exhaustive search possible)\n")
display(cmp_tab.style.apply(hl, axis=1).format({"Return %": "{:.2f}", "Vol %": "{:.2f}", "Sharpe": "{:.3f}", "Objective": "{:.6f}",
                                                 "Gap to best known": "{:.2e}", "Gap to MIQP bound": "{:.2e}"}, na_rep="-"))

# repeated experiments: share of feasible reads at the best known objective
def at_best(label):
    lst = EXP_E[label]; tot = sum(len(e) for e in lst)
    return (sum(int(np.isclose(e, F_BEST, atol=1e-9).sum()) for e in lst) / max(tot, 1) * 100) if tot else 0.0
stat = pd.concat([tab_main, tab_own])
stat["Gap of best objective to best known (mean over seeds)"] = stat["Best objective (mean over seeds)"] - F_BEST
names_exp = ["Ising D-Wave", "QUBO D-Wave"] + list(rows_own.keys())
stat["Feasible reads at best known (%)"] = [at_best(n_) for n_ in names_exp]
print("\nSolver statistics of the repeated experiments (Sections 8.5a and 8.6):")
display(stat.round({"Best objective (mean over seeds)": 6, "Best objective (best seed)": 6, "Best objective (worst seed)": 6, "Mean objective of feasible reads": 6,
                    "Feasible reads (%)": 1, "Runtime per run (s)": 2, "Gap of best objective to best known (mean over seeds)": 8, "Feasible reads at best known (%)": 2}))
stat.to_csv(f"{OUT_DIR}/solver_statistics_500stocks.csv")

### 9.2 Conclusions

The next cell writes the conclusions. **Every number in the text is filled in from the results above**, so the statement always matches the run.

In [ ]:
# SECTION 9.2 | conclusions (every number is filled in from the results computed above)
mi, mq = tab_main.iloc[0], tab_main.iloc[1]
o_i, o_q = tab_own.iloc[0], tab_own.iloc[2]
o_p = tab_own.iloc[1]
r_mq_, v_mq_, s_mq_, sh_mq_ = port_stats(w_miqp)
best_label = min(((m_, obj_w(w_)) for m_, w_ in methods[1:]), key=lambda t: t[1])
gap_rel = (MIQP_OBJ - MIQP_BOUND) / abs(MIQP_OBJ)
sw_first, sw_last = tab_sweeps.iloc[0], tab_sweeps.iloc[-1]
P_lo, P_1, P_hi = tab_P.iloc[0], tab_P.iloc[1], tab_P.iloc[2]
frus_txt = (f"{frustrated:,} of the {n_tri:,} triangles are frustrated" if frustrated is not None else "the number of frustrated triangles was not computed")

display(Markdown(f"""
**Statement on the compared results ({N} S&P 500 stocks with full history, {returns.index[0].date()} to {returns.index[-1].date()}; k = {K}, lambda = {LAMBDA:.4f}, P = {P:.5f}).**
The lowest objective found by any equal-weight solver (QUBO or Ising) is **{F_BEST:.6f}** (best known, not proven optimal; exhaustive search over C({N},{K}) = {comb(N, K):.2e} portfolios is impossible). The MIQP with free weights has objective {MIQP_OBJ:.6f} with solver lower bound {MIQP_BOUND:.6f} ({SOLVER}, {status}, relative gap {gap_rel:.1e}, {t_miqp:.0f} s); it holds {len(held)} stocks with return {r_mq_ * 100:.2f}%, volatility {v_mq_ ** 0.5 * 100:.2f}%, Sharpe {sh_mq_:.3f}. Best known equal-weight objective minus MIQP objective = {F_BEST - MIQP_OBJ:+.2e}; minus MIQP lower bound = {F_BEST - MIQP_BOUND:+.2e} (the lower bound is valid for the equal-weight problem). {'A negative first value means that the MIQP solution found within the time limit is worse than the best equal-weight solution, so the MIQP did not reach its optimum (relative gap ' + format(gap_rel, '.1e') + '); raise MIQP_TIME_LIMIT or use a full Gurobi licence.' if F_BEST < MIQP_OBJ else 'The difference to the MIQP objective contains both the weight model (equal weights against free weights) and the remaining search gap of the annealers.'} In the main runs the best equal-weight result was {best_label[1]:.6f} ({best_label[0]}).

**1. Ising and QUBO are the same problem.** The hand-coded Ising model agrees with the `dimod` conversion (differences below 1e-10), the Ising energy equals the QUBO energy on 200,000 random spin states (max relative difference {max_rel:.1e}) and equals the Markowitz objective on 5,000 random states with exactly {K} stocks (max difference {max_diff:.1e}). The two models differ only in the variables ({N} bits against {N} spins) and the fixed magnetisation sum(s) = {M_FIXED}.

**2. The spin-glass picture.** {(Jp > 0).sum():,} of the {len(iu[0]):,} couplings J_ij are positive (antiferromagnetic) and {(h > 0).sum()} of the {N} local fields h_i are positive; {frus_txt}, on a fully connected graph, with the constraint "exactly k stocks" acting as the fixed magnetisation. The penalty alone contributes P/2 = {J_pen:.6f} to every coupling, i.e. {J_pen / Jp.mean() * 100:.0f}% of the mean coupling ({Jp.mean():.6f}); the covariance part ranges over [{J_cov[iu].min():.6f}, {J_cov[iu].max():.6f}] and is the only source of differences between the couplings (std of J_ij = {Jp.std():.6f}).

**3. Solver behaviour (D-Wave sampler, {EXP_READS} reads x {EXP_SWEEPS} sweeps, {N_SEEDS} seeds).** The best feasible objective per seed was {mi["Best objective (mean over seeds)"]:.6f} on average for the Ising model (best seed {mi["Best objective (best seed)"]:.6f}, worst {mi["Best objective (worst seed)"]:.6f}) and {mq["Best objective (mean over seeds)"]:.6f} for the QUBO model (best {mq["Best objective (best seed)"]:.6f}, worst {mq["Best objective (worst seed)"]:.6f}); the share of reads with exactly {K} stocks was {mi["Feasible reads (%)"]:.1f}% (Ising) and {mq["Feasible reads (%)"]:.1f}% (QUBO); runtime per run {mi["Runtime per run (s)"]:.1f} s (Ising) and {mq["Runtime per run (s)"]:.1f} s (QUBO). In the main run ({NUM_READS} x {NUM_SWEEPS}, seed {SEED}) the best objective was {FOUND["Ising + D-Wave SA sampler"]:.6f} (Ising) and {FOUND["QUBO + D-Wave SA sampler"]:.6f} (QUBO).
The own annealers ({SA_CHAINS} x {SA_SWEEPS}, {N_SEEDS} seeds) gave an average best objective of {o_i["Best objective (mean over seeds)"]:.6f} (Ising, single flips, {o_i["Feasible reads (%)"]:.1f}% feasible reads), {o_p["Best objective (mean over seeds)"]:.6f} (Ising with two-spin moves, {o_p["Feasible reads (%)"]:.1f}% feasible) and {o_q["Best objective (mean over seeds)"]:.6f} (QUBO with single and two-bit moves, {o_q["Feasible reads (%)"]:.1f}% feasible). The Ising annealer with two-spin moves and the QUBO annealer use the same moves and budget: their average best objectives are {o_p["Best objective (mean over seeds)"]:.6f} and {o_q["Best objective (mean over seeds)"]:.6f} (difference {abs(o_p["Best objective (mean over seeds)"] - o_q["Best objective (mean over seeds)"]):.1e}); the Ising annealer without two-spin moves reached {o_i["Best objective (mean over seeds)"]:.6f}.

**4. Sensitivity (Ising, D-Wave sampler).** With {int(sw_first.name)} sweeps the best objective was {sw_first["Best objective (mean over seeds)"]:.6f} with {sw_first["Feasible reads (%)"]:.1f}% feasible reads; with {int(sw_last.name)} sweeps {sw_last["Best objective (mean over seeds)"]:.6f} with {sw_last["Feasible reads (%)"]:.1f}%. For the penalty 0.5x / 1x / 2x of the Monte-Carlo value the best objective was {P_lo["Best objective (mean over seeds)"]:.6f} / {P_1["Best objective (mean over seeds)"]:.6f} / {P_hi["Best objective (mean over seeds)"]:.6f} and the share of feasible reads {P_lo["Feasible reads (%)"]:.1f}% / {P_1["Feasible reads (%)"]:.1f}% / {P_hi["Feasible reads (%)"]:.1f}%.

**5. Scale.** At N = {N} the exact optimum of the equal-weight problem cannot be found by enumeration, so a solver can only be judged by its distance to the best known value and to the MIQP bound (table in Section 9.1). All annealing results above are single-machine classical simulations; the Ising model was **not** run on a quantum annealer.
"""))